In [ ]:
from __future__ import annotations

import copy, gc, hashlib, json, math, os, platform, random, shutil, subprocess, sys, time, warnings, zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from matplotlib import pyplot as plt
from scipy import stats
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet18
from torchvision.transforms import v2
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

try:
    import google.colab  
    RUN_IN_COLAB = True
except Exception:
    RUN_IN_COLAB = False

USE_GOOGLE_DRIVE = os.environ.get("RTTP_USE_GOOGLE_DRIVE", "1") != "0"
DRIVE_BASE = Path(os.environ.get("RTTP_DRIVE_BASE", "/content/drive/MyDrive"))
ROOT_NAME = os.environ.get("RTTP_ROOT_NAME", "optimizer_response_cifar10_rttp")

if RUN_IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = DRIVE_BASE / ROOT_NAME
else:
    ROOT = Path.cwd() / ROOT_NAME

FIG = ROOT / "figures"
TAB = ROOT / "tables"
CACHE = ROOT / "cache"
CHK = CACHE / "checkpoints"
RESPONSE_CHUNKS = CHK / "response_grid"
RESPONSE_TENSORS = CHK / "response_tensors"
SPECTRUM_CHUNKS = CHK / "response_spectrum"
POLICY_CHUNKS = CHK / "policy"
ARCH = ROOT / "archives"
for pth in (ROOT, FIG, TAB, CACHE, CHK, RESPONSE_CHUNKS, RESPONSE_TENSORS, SPECTRUM_CHUNKS, POLICY_CHUNKS, ARCH):
    pth.mkdir(parents=True, exist_ok=True)

FORCE_RECOMPUTE = os.environ.get("RTTP_FORCE_RECOMPUTE", "0") == "1"
FORCE_RESPONSE_GRID = FORCE_RECOMPUTE or os.environ.get("RTTP_FORCE_RESPONSE_GRID", "0") == "1"
FORCE_APPLICATION = FORCE_RECOMPUTE or os.environ.get("RTTP_FORCE_APPLICATION", "0") == "1"
USE_BATCHED_HVP = os.environ.get("RTTP_USE_BATCHED_HVP", "1") != "0"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
print("output root:", ROOT)
print("batched HVP enabled:", USE_BATCHED_HVP)


def format_duration(seconds: float) -> str:
    seconds=max(0,int(round(float(seconds))))
    h,rem=divmod(seconds,3600); m,s=divmod(rem,60)
    if h: return f"{h}h {m:02d}m {s:02d}s"
    if m: return f"{m}m {s:02d}s"
    return f"{s}s"


class EtaTimer:
    def __init__(self,total:int,label:str):
        self.total=max(int(total),0); self.label=label; self.start=time.perf_counter(); self.done=0
    def update(self,done=None,n:int=1,note:str=""):
        self.done = int(done) if done is not None else self.done + int(n)
        elapsed=time.perf_counter()-self.start
        rate=self.done/elapsed if elapsed>0 and self.done>0 else float("nan")
        remaining=(self.total-self.done)/rate if np.isfinite(rate) and rate>0 else float("nan")
        suffix=f" | {note}" if note else ""
        eta=format_duration(remaining) if np.isfinite(remaining) else "unknown"
        print(f"[{self.label}] {self.done}/{self.total} done | elapsed {format_duration(elapsed)} | ETA {eta}{suffix}")
    def finish(self,note:str=""):
        self.update(done=self.total,note=note)


def timed_block(label:str):
    class _TimedBlock:
        def __enter__(self):
            self.t0=time.perf_counter(); print(f"[{label}] start"); return self
        def __exit__(self,exc_type,exc,tb):
            status="failed" if exc_type else "done"
            print(f"[{label}] {status} in {format_duration(time.perf_counter()-self.t0)}")
            return False
    return _TimedBlock()


def _atomic_replace(tmp_path: Path, final_path: Path) -> None:
    final_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path.replace(final_path)


def atomic_write_text(path: Path, text: str) -> None:
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text)
    _atomic_replace(tmp, path)


def table_paths(stem: Path) -> Tuple[Path, Path]:
    stem = Path(stem)
    return Path(str(stem) + ".csv"), Path(str(stem) + ".parquet")


def write_dataframe(df: pd.DataFrame, stem: Path) -> None:
    csv_path, parquet_path = table_paths(stem)
    tmp_csv = csv_path.with_name(csv_path.name + ".tmp")
    df.to_csv(tmp_csv, index=False)
    _atomic_replace(tmp_csv, csv_path)
    try:
        tmp_parquet = parquet_path.with_name(parquet_path.name + ".tmp")
        df.to_parquet(tmp_parquet, index=False)
        _atomic_replace(tmp_parquet, parquet_path)
    except Exception as e:
        print(f"parquet write skipped for {parquet_path.name}: {e}")


def read_dataframe(stem: Path):
    csv_path, parquet_path = table_paths(stem)
    if parquet_path.exists():
        try:
            return pd.read_parquet(parquet_path)
        except Exception as e:
            print(f"parquet read failed for {parquet_path.name}; trying CSV: {e}")
    if csv_path.exists():
        return pd.read_csv(csv_path)
    return None


def float_tag(x: float) -> str:
    return f"{float(x):.8g}".replace("-", "m").replace(".", "p").replace("+", "")


def optimizer_tag(optimizer: str, beta2: float | None = None) -> str:
    if optimizer == "heavy_ball":
        return "heavy_ball"
    tag = optimizer
    if beta2 is not None and np.isfinite(beta2):
        tag += f"__b2={float_tag(beta2)}"
    return tag


def memory_component_tag(memory_component: str | None = None) -> str:
    if memory_component is None:
        return "memory"
    return str(memory_component).replace("/", "_").replace(" ", "_")


def response_chunk_stem(arch: str, seed: int, lr: float, mu: float, H: int, optimizer: str = "heavy_ball", beta2: float | None = None, memory_component: str = "momentum") -> Path:
    
    if optimizer == "heavy_ball" and memory_component in ("momentum", "memory", None):
        return RESPONSE_CHUNKS / f"response__arch={arch}__seed={seed}__lr={float_tag(lr)}__mu={float_tag(mu)}__H={H}"
    return RESPONSE_CHUNKS / f"response__opt={optimizer_tag(optimizer,beta2)}__mem={memory_component_tag(memory_component)}__arch={arch}__seed={seed}__lr={float_tag(lr)}__b1={float_tag(mu)}__H={H}"


def spectrum_chunk_stem(arch: str, seed: int, lr: float, mu: float, H: int, optimizer: str = "heavy_ball", beta2: float | None = None, memory_component: str = "momentum") -> Path:
    if optimizer == "heavy_ball" and memory_component in ("momentum", "memory", None):
        return SPECTRUM_CHUNKS / f"spectrum__arch={arch}__seed={seed}__lr={float_tag(lr)}__mu={float_tag(mu)}__H={H}"
    return SPECTRUM_CHUNKS / f"spectrum__opt={optimizer_tag(optimizer,beta2)}__mem={memory_component_tag(memory_component)}__arch={arch}__seed={seed}__lr={float_tag(lr)}__b1={float_tag(mu)}__H={H}"


def response_tensor_path(arch: str, seed: int, lr: float, mu: float, H: int, optimizer: str = "heavy_ball", beta2: float | None = None, memory_component: str = "momentum") -> Path:
    if optimizer == "heavy_ball" and memory_component in ("momentum", "memory", None):
        return RESPONSE_TENSORS / f"rttp_response__arch={arch}__seed={seed}__lr={float_tag(lr)}__mu={float_tag(mu)}__H={H}.pt"
    return RESPONSE_TENSORS / f"rttp_response__opt={optimizer_tag(optimizer,beta2)}__mem={memory_component_tag(memory_component)}__arch={arch}__seed={seed}__lr={float_tag(lr)}__b1={float_tag(mu)}__H={H}.pt"


def policy_chunk_stem(arch: str, seed: int, optimizer: str = "heavy_ball", lr: float | None = None, mu: float | None = None, beta2: float | None = None, memory_component: str = "momentum") -> Path:
    if optimizer == "heavy_ball" and memory_component in ("momentum", "memory", None):
        return POLICY_CHUNKS / f"policy__arch={arch}__seed={seed}"
    return POLICY_CHUNKS / f"policy__opt={optimizer_tag(optimizer,beta2)}__mem={memory_component_tag(memory_component)}__arch={arch}__seed={seed}__lr={float_tag(lr)}__b1={float_tag(mu)}"


def cpu_state(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu()
    return [cpu_state(v) for v in x]


def device_state(x, device=DEVICE):
    if isinstance(x, torch.Tensor):
        return x.to(device)
    return [device_state(v, device) for v in x]


def torch_save_atomic(obj, path: Path) -> None:
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    _atomic_replace(tmp, path)


In [ ]:
@dataclass(frozen=True)
class Config:
    run_mode: str = "paper"  
    dataset_id: str = "uoft-cs/cifar10"
    architectures: Tuple[str, ...] = ("smallcnn", "resnet18")
    optimizers: Tuple[str, ...] = ("heavy_ball", "adam", "adamw")
    adaptive_memory_components: Tuple[str, ...] = ("m", "log_v", "joint")
    application_optimizers: Tuple[str, ...] = ("heavy_ball",)
    boundary_classes: Tuple[int, ...] = (0, 1, 2, 3, 4)
    continuation_classes: Tuple[int, ...] = (5, 6, 7, 8, 9)
    train_per_class: int = 1000
    test_per_class: int = 500
    batch_size: int = 128
    boundary_epochs_smallcnn: int = 12
    boundary_epochs_resnet18: int = 20
    boundary_lr_smallcnn: float = 0.03
    boundary_lr_resnet18: float = 0.03
    boundary_lr_adam_smallcnn: float = 1e-3
    boundary_lr_adam_resnet18: float = 1e-3
    boundary_momentum: float = 0.9
    weight_decay: float = 0.0
    seeds: Tuple[int, ...] = (0, 1, 2, 3, 4)
    horizons: Tuple[int, ...] = (1, 5, 10, 20, 40)
    response_lrs: Tuple[float, ...] = (0.01, 0.03)
    response_momenta: Tuple[float, ...] = (0.5, 0.9)
    adam_response_lrs: Tuple[float, ...] = (3e-4, 1e-3)
    adam_beta1s: Tuple[float, ...] = (0.9,)
    adam_beta2: float = 0.999
    adam_eps: float = 1e-8
    adam_v_floor: float = 1e-12
    
    finite_difference_scales: Tuple[float, ...] = (3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1)
    response_rank: int = 8
    n_probe_directions: int = 8
    rank_sweep: Tuple[int, ...] = (1, 2, 4, 8)
    probe_size: int = 512
    num_workers: int = 2
    use_amp_boundary: bool = False
    run_audit: bool = True
    run_application: bool = True
    certificate_tradeoffs: Tuple[float, ...] = (0.25, 0.5, 1.0, 2.0, 4.0)
    certificate_radius_scales: Tuple[float, ...] = (0.25, 0.5, 1.0)

CFG = Config()

if CFG.run_mode == "smoke":
    CFG = Config(run_mode="smoke", architectures=("smallcnn",), optimizers=("heavy_ball","adam","adamw"),
                 adaptive_memory_components=("m","log_v","joint"),
                 application_optimizers=("heavy_ball",), train_per_class=100,
                 test_per_class=100, boundary_epochs_smallcnn=1, seeds=(0,), horizons=(1, 3),
                 response_lrs=(0.01,), response_momenta=(0.9,), adam_response_lrs=(1e-3,), adam_beta1s=(0.9,),
                 finite_difference_scales=(1e-3, 1e-2), response_rank=2, n_probe_directions=2,
                 rank_sweep=(1,2), probe_size=64, run_application=True)
elif CFG.run_mode == "development":
    CFG = Config(run_mode="development", adaptive_memory_components=("m","log_v","joint"), train_per_class=500, test_per_class=250,
                 boundary_epochs_smallcnn=5, boundary_epochs_resnet18=8, seeds=(0,1),
                 horizons=(1,5,20), response_lrs=(0.01,), response_momenta=(0.5,0.9),
                 adam_response_lrs=(1e-3,), adam_beta1s=(0.9,), finite_difference_scales=(1e-3, 1e-2, 1e-1),
                 response_rank=4, n_probe_directions=4, rank_sweep=(1,2,4), probe_size=256)

atomic_write_text(ROOT / "config.json", json.dumps(asdict(CFG), indent=2))
CFG


In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

class HFCifarTask(Dataset):
    def __init__(self, hf_split, classes: Sequence[int], per_class: int, train: bool):
        self.classes = tuple(classes)
        self.label_map = {c:i for i,c in enumerate(self.classes)}
        counts = {c:0 for c in self.classes}
        rows=[]
        for row in hf_split:
            y=int(row["label"])
            if y in counts and counts[y] < per_class:
                rows.append(row); counts[y]+=1
            if all(v >= per_class for v in counts.values()): break
        self.rows=rows
        self.transform = (v2.Compose([
            v2.ToImage(), v2.RandomCrop(32, padding=4), v2.RandomHorizontalFlip(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616)),
        ]) if train else v2.Compose([
            v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
            v2.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616)),
        ]))
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        row=self.rows[i]
        return self.transform(row["img"]), self.label_map[int(row["label"])]

def make_data(seed: int):
    seed_everything(seed)
    ds=load_dataset(CFG.dataset_id)
    A_tr=HFCifarTask(ds["train"], CFG.boundary_classes, CFG.train_per_class, True)
    B_tr=HFCifarTask(ds["train"], CFG.continuation_classes, CFG.train_per_class, True)
    A_te=HFCifarTask(ds["test"], CFG.boundary_classes, CFG.test_per_class, False)
    B_te=HFCifarTask(ds["test"], CFG.continuation_classes, CFG.test_per_class, False)
    gen=torch.Generator().manual_seed(seed)
    kw=dict(batch_size=CFG.batch_size, num_workers=CFG.num_workers, pin_memory=torch.cuda.is_available())
    return {
        "A_train": DataLoader(A_tr, shuffle=True, generator=gen, **kw),
        "B_train": DataLoader(B_tr, shuffle=True, generator=gen, **kw),
        "A_test": DataLoader(A_te, shuffle=False, **kw),
        "B_test": DataLoader(B_te, shuffle=False, **kw),
    }

def materialize_batches(loader: DataLoader, H: int, device=DEVICE):
    out=[]; it=iter(loader)
    while len(out)<H:
        try: x,y=next(it)
        except StopIteration: it=iter(loader); x,y=next(it)
        out.append((x.to(device), y.to(device)))
    return out

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.ReLU(), nn.Conv2d(32,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.Conv2d(64,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1))
        self.fc=nn.Linear(128,n_classes)
    def forward(self,x): return self.fc(self.features(x).flatten(1))

def build_model(name: str) -> nn.Module:
    if name=="smallcnn": return SmallCNN(5)
    if name=="resnet18":
        m=resnet18(weights=None, num_classes=5)
        m.conv1=nn.Conv2d(3,64,kernel_size=3,stride=1,padding=1,bias=False)
        m.maxpool=nn.Identity()
        return m
    raise ValueError(name)

def n_parameters(model): return sum(p.numel() for p in model.parameters())
for arch in CFG.architectures:
    m=build_model(arch); print(arch, f"{n_parameters(m):,}"); del m

In [ ]:
def params_list(model): return [p for p in model.parameters() if p.requires_grad]
def zeros_like_params(model): return [torch.zeros_like(p) for p in params_list(model)]
def clone_state(xs):
    if isinstance(xs, torch.Tensor):
        return xs.detach().clone()
    return [clone_state(x) for x in xs]
def dot_state(a,b): return sum((x*y).sum() for x,y in zip(a,b))
def norm_state(a): return torch.sqrt(torch.clamp(dot_state(a,a), min=0.0))
def add_state(a,b,alpha=1.0): return [x+alpha*y for x,y in zip(a,b)]
def scale_state(a,c): return [c*x for x in a]
def flatten_state(a): return torch.cat([x.reshape(-1) for x in a])
def unflatten_like(v, ref):
    out=[]; i=0
    for x in ref:
        out.append(v[i:i+x.numel()].view_as(x)); i+=x.numel()
    assert i==v.numel(); return out

def set_params_(model, values):
    with torch.no_grad():
        for p,v in zip(params_list(model), values): p.copy_(v)

def loss_and_grads(model,x,y,create_graph=False):
    loss=F.cross_entropy(model(x),y)
    g=torch.autograd.grad(loss, params_list(model), create_graph=create_graph)
    return loss, list(g)

def optimizer_settings():
    rows=[]
    if "heavy_ball" in CFG.optimizers:
        for lr in CFG.response_lrs:
            for mu in CFG.response_momenta:
                rows.append(dict(optimizer="heavy_ball",learning_rate=lr,momentum=mu,beta2=np.nan,
                                 memory_component="momentum",state_parameterization="additive",
                                 nominal_memory="reset"))
    for opt in CFG.optimizers:
        if opt in ("adam","adamw"):
            for lr in CFG.adam_response_lrs:
                for b1 in CFG.adam_beta1s:
                    for comp in CFG.adaptive_memory_components:
                        param="additive" if comp=="m" else "log" if comp=="log_v" else "additive+log"
                        rows.append(dict(optimizer=opt,learning_rate=lr,momentum=b1,beta2=CFG.adam_beta2,
                                         memory_component=comp,state_parameterization=param,
                                         nominal_memory="inherited"))
    return rows

def memory_zeros_like(model, optimizer="heavy_ball"):
    z=zeros_like_params(model)
    if optimizer=="heavy_ball": return z
    if optimizer in ("adam","adamw"): return z + zeros_like_params(model)
    raise ValueError(optimizer)

def split_memory(memory, model, optimizer="heavy_ball"):
    n=len(params_list(model))
    if optimizer=="heavy_ball": return memory, None
    if len(memory)==2 and isinstance(memory[0], (list, tuple)) and isinstance(memory[1], (list, tuple)):
        return list(memory[0]), list(memory[1])
    return memory[:n], memory[n:]

def normalize_memory_state(memory, model, optimizer="heavy_ball"):
    n=len(params_list(model))
    if optimizer=="heavy_ball":
        if len(memory)==1 and isinstance(memory[0], (list, tuple)):
            memory=list(memory[0])
        if len(memory)!=n:
            raise ValueError(f"heavy_ball memory has {len(memory)} tensors, expected {n}")
        return clone_state(memory)
    if optimizer in ("adam","adamw"):
        if len(memory)==2 and isinstance(memory[0], (list, tuple)) and isinstance(memory[1], (list, tuple)):
            memory=list(memory[0])+list(memory[1])
        if len(memory)!=2*n:
            raise ValueError(f"{optimizer} memory has {len(memory)} tensors, expected {2*n}; expected flat m+v or nested [m,v]")
        return clone_state(memory)
    raise ValueError(optimizer)

def memory_template_like(model, optimizer="heavy_ball"):
    return memory_zeros_like(model, optimizer)

def adam_bias_correction(step, beta):
    return 1.0 - beta**int(step)

def optimizer_step_(model, memory, x, y, optimizer, lr, momentum, beta2=np.nan, step=1, wd=0.0):
    loss,g=loss_and_grads(model,x,y,False)
    params=params_list(model)
    with torch.no_grad():
        if optimizer=="heavy_ball":
            for i,(p,gi) in enumerate(zip(params,g)):
                if wd: gi=gi+wd*p
                memory[i].mul_(momentum).add_(gi)
                p.add_(memory[i],alpha=-lr)
        elif optimizer in ("adam","adamw"):
            m,v=split_memory(memory,model,optimizer)
            b1=float(momentum); b2=float(beta2); eps=CFG.adam_eps
            c1=adam_bias_correction(step,b1); c2=adam_bias_correction(step,b2)
            for i,(p,gi) in enumerate(zip(params,g)):
                if optimizer=="adam" and wd: gi=gi+wd*p
                m[i].mul_(b1).add_(gi,alpha=1-b1)
                v[i].mul_(b2).addcmul_(gi,gi,value=1-b2)
                mhat=m[i]/c1; vhat=v[i]/c2
                if optimizer=="adamw" and wd: p.mul_(1-lr*wd)
                p.addcdiv_(mhat, torch.sqrt(torch.clamp(vhat,min=eps*eps)).add(eps), value=-lr)
        else:
            raise ValueError(optimizer)
    return float(loss.detach())


def heavy_ball_step_(model, velocity, x, y, lr, mu, wd=0.0):
    return optimizer_step_(model, velocity, x, y, "heavy_ball", lr, mu, np.nan, 1, wd)

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); n=0; correct=0; loss=0.0
    for x,y in loader:
        x,y=x.to(DEVICE),y.to(DEVICE); logits=model(x)
        loss+=F.cross_entropy(logits,y,reduction="sum").item(); correct+=(logits.argmax(1)==y).sum().item(); n+=y.numel()
    model.train(); return {"loss":loss/n,"acc":correct/n}


In [ ]:
def boundary_hparams(arch, optimizer="heavy_ball"):
    epochs = CFG.boundary_epochs_smallcnn if arch=="smallcnn" else CFG.boundary_epochs_resnet18
    if optimizer=="heavy_ball":
        lr = CFG.boundary_lr_smallcnn if arch=="smallcnn" else CFG.boundary_lr_resnet18
    elif optimizer in ("adam","adamw"):
        lr = CFG.boundary_lr_adam_smallcnn if arch=="smallcnn" else CFG.boundary_lr_adam_resnet18
    else:
        raise ValueError(optimizer)
    return epochs, lr

def boundary_cache_paths(arch, seed, optimizer="heavy_ball"):
    if optimizer=="heavy_ball":
        return CACHE/f"boundary_{arch}_seed{seed}.pt", CACHE/f"boundary_{arch}_seed{seed}_partial.pt"
    return CACHE/f"boundary_{optimizer}_{arch}_seed{seed}.pt", CACHE/f"boundary_{optimizer}_{arch}_seed{seed}_partial.pt"

def deterministic_epoch_loader(base_loader: DataLoader, seed: int, epoch: int) -> DataLoader:
    gen=torch.Generator().manual_seed(seed*1_000_003 + epoch)
    return DataLoader(base_loader.dataset, batch_size=base_loader.batch_size, shuffle=True,
                      generator=gen, num_workers=base_loader.num_workers,
                      pin_memory=torch.cuda.is_available())

def boundary_meta_from_checkpoint(ck, data=None):
    hist=ck.get("metrics",[])
    if "step_count" in ck:
        step_count=int(ck["step_count"])
    elif data is not None and "A_train" in data:
        step_count=int(len(hist)*len(data["A_train"]))
    else:
        step_count=0
    return {"metrics":hist,"step_count":step_count,"optimizer":ck.get("optimizer","heavy_ball")}

def train_or_load_boundary(arch, seed, data, optimizer="heavy_ball"):
    path,partial_path=boundary_cache_paths(arch,seed,optimizer)
    model=build_model(arch).to(DEVICE)
    if path.exists():
        ck=torch.load(path,map_location=DEVICE)
        set_params_(model,ck["theta"])
        memory=ck["memory"] if "memory" in ck else ck["velocity"]
        memory=normalize_memory_state(memory,model,optimizer)
        return model,memory,boundary_meta_from_checkpoint(ck,data)

    seed_everything(seed); epochs,lr=boundary_hparams(arch,optimizer); hist=[]
    memory=memory_zeros_like(model,optimizer); start_epoch=0; step_count=0
    if partial_path.exists():
        ck=torch.load(partial_path,map_location=DEVICE)
        set_params_(model,ck["theta"]); memory=ck.get("memory",ck.get("velocity")); hist=ck.get("metrics",[])
        memory=normalize_memory_state(memory,model,optimizer)
        step_count=int(ck.get("step_count",len(hist)*len(data["A_train"])))
        start_epoch=len(hist)
        print(f"resuming boundary {optimizer} {arch} seed {seed} from epoch {start_epoch}")

    epoch_timer=EtaTimer(epochs-start_epoch,f"boundary {optimizer} {arch} seed={seed}")
    for epoch in range(start_epoch, epochs):
        seed_everything(seed*1_000_003 + epoch)
        t_epoch=time.perf_counter()
        for x,y in deterministic_epoch_loader(data["A_train"],seed,epoch):
            x,y=x.to(DEVICE),y.to(DEVICE); step_count+=1
            beta2 = CFG.adam_beta2 if optimizer in ("adam","adamw") else np.nan
            momentum = CFG.boundary_momentum if optimizer=="heavy_ball" else CFG.adam_beta1s[0]
            optimizer_step_(model,memory,x,y,optimizer,lr,momentum,beta2,step_count,CFG.weight_decay)
        metrics=evaluate(model,data["A_test"]); hist.append({"epoch":epoch+1,**metrics}); print(optimizer,arch,seed,hist[-1])
        partial_ck={"theta":clone_state(params_list(model)),"memory":clone_state(memory),"metrics":hist,"optimizer":optimizer,"step_count":step_count}
        torch.save(partial_ck,partial_path)
        epoch_timer.update(note=f"epoch={epoch+1}, acc={metrics['acc']:.3f}, epoch_time={format_duration(time.perf_counter()-t_epoch)}")
    ck={"theta":clone_state(params_list(model)),"memory":clone_state(memory),"metrics":hist,"optimizer":optimizer,"step_count":step_count}
    torch.save(ck,path); return model,memory,boundary_meta_from_checkpoint(ck,data)


In [ ]:
def state_zeros_like(ref):
    return [torch.zeros_like(x) for x in ref]

def clamp_state_min(xs, floor):
    return [torch.clamp(x.detach().clone(), min=float(floor)) for x in xs]

def adam_memory_parts(memory, model):
    memory=normalize_memory_state(memory,model,"adam")
    m,v=split_memory(memory,model,"adam")
    return clone_state(m), clone_state(v)

def adam_v_base(memory, model):
    _,v=adam_memory_parts(memory,model)
    return clamp_state_min(v, CFG.adam_v_floor)

def adam_memory_from_parts(m, v):
    return clone_state(m)+clone_state(v)

def embed_adam_coordinate(model, dm=None, zlogv=None):
    z=zeros_like_params(model)
    dm_part=clone_state(dm) if dm is not None else state_zeros_like(z)
    zlog_part=clone_state(zlogv) if zlogv is not None else state_zeros_like(z)
    return dm_part+zlog_part

def embed_adam_tangent(model, vbase, dm=None, zlogv=None):
    z=zeros_like_params(model)
    dm_part=clone_state(dm) if dm is not None else state_zeros_like(z)
    dv_part=[vb*zl for vb,zl in zip(vbase,zlogv)] if zlogv is not None else state_zeros_like(z)
    return dm_part+dv_part

def orthonormal_basis_from_candidates(candidates, rank, seed, random_candidate_fn):
    seed_everything(seed+9187); vecs=[]
    for cand in candidates:
        if cand is None: continue
        v=clone_state(cand)
        for q in vecs: v=add_state(v,q,alpha=-float(dot_state(v,q)))
        n=float(norm_state(v))
        if n>1e-8: vecs.append(scale_state(v,1/n))
        if len(vecs)>=rank: break
    while len(vecs)<rank:
        v=random_candidate_fn()
        for q in vecs: v=add_state(v,q,alpha=-float(dot_state(v,q)))
        n=float(norm_state(v))
        if n>1e-8: vecs.append(scale_state(v,1/n))
    return vecs[:rank]

def make_component_basis(boundary_model, inherited, optimizer, memory_component, rank, seed):
    if optimizer=="heavy_ball":
        template=memory_template_like(boundary_model,optimizer)
        first=scale_state(inherited,1.0) if norm_state(inherited).item()>1e-12 else None
        coord=orthonormal_basis_from_candidates([first],rank,seed,lambda:[torch.randn_like(x) for x in template])
        return {"coord":coord,"tangent":coord,"rank":len(coord),"memory_component":"momentum",
                "state_parameterization":"additive","nominal_memory":"reset"}
    if optimizer not in ("adam","adamw"):
        raise ValueError(optimizer)
    if memory_component not in ("m","log_v","joint"):
        raise ValueError(memory_component)
    m0,v0=adam_memory_parts(inherited,boundary_model); vbase=clamp_state_min(v0,CFG.adam_v_floor)
    ztemplate=zeros_like_params(boundary_model)
    if memory_component=="joint":
        m_basis=make_component_basis(boundary_model,inherited,optimizer,"m",rank,seed)
        v_basis=make_component_basis(boundary_model,inherited,optimizer,"log_v",rank,seed+31_337)
        return {"coord":m_basis["coord"]+v_basis["coord"],"tangent":m_basis["tangent"]+v_basis["tangent"],
                "rank":len(m_basis["coord"])+len(v_basis["coord"]),"memory_component":"joint",
                "state_parameterization":"additive+log","nominal_memory":"inherited"}
    if memory_component=="m":
        first=embed_adam_coordinate(boundary_model,dm=m0) if norm_state(m0).item()>1e-12 else None
        coord=orthonormal_basis_from_candidates([first],rank,seed,
            lambda: embed_adam_coordinate(boundary_model,dm=[torch.randn_like(x) for x in ztemplate]))
        tangent=clone_state(coord)
        return {"coord":coord,"tangent":tangent,"rank":len(coord),"memory_component":"m",
                "state_parameterization":"additive","nominal_memory":"inherited"}
    ones=[torch.ones_like(x) for x in ztemplate]
    coord=orthonormal_basis_from_candidates([embed_adam_coordinate(boundary_model,zlogv=ones)],rank,seed,
        lambda: embed_adam_coordinate(boundary_model,zlogv=[torch.randn_like(x) for x in ztemplate]))
    tangent=[]
    for c in coord:
        _,zlog=split_memory(c,boundary_model,optimizer)
        tangent.append(embed_adam_tangent(boundary_model,vbase,zlogv=zlog))
    return {"coord":coord,"tangent":tangent,"rank":len(coord),"memory_component":"log_v",
            "state_parameterization":"log","nominal_memory":"inherited"}


def make_basis(template, inherited, rank, seed):
    seed_everything(seed+9187); vecs=[]
    if norm_state(inherited).item()>1e-12: vecs.append(scale_state(inherited,1.0/norm_state(inherited)))
    while len(vecs)<rank:
        v=[torch.randn_like(x) for x in template]
        for q in vecs: v=add_state(v,q,alpha=-float(dot_state(v,q)))
        n=float(norm_state(v))
        if n>1e-8: vecs.append(scale_state(v,1/n))
    return vecs[:rank]

def combine_basis(B, alpha):
    out=[torch.zeros_like(x) for x in B[0]]
    for a,b in zip(alpha,B): out=add_state(out,b,alpha=float(a))
    return out

def response_nominal_memory(boundary_model, inherited, optimizer, memory_component):
    if optimizer=="heavy_ball":
        return memory_zeros_like(boundary_model,optimizer), {"nominal_memory":"reset","state_parameterization":"additive","memory_component":"momentum"}
    m0,v0=adam_memory_parts(inherited,boundary_model)
    vbase=clamp_state_min(v0,CFG.adam_v_floor)
    param="additive" if memory_component=="m" else "log" if memory_component=="log_v" else "additive+log"
    return adam_memory_from_parts(m0,vbase), {"nominal_memory":"inherited","state_parameterization":param,"memory_component":memory_component}

def memory_from_coordinate(boundary_model, nominal_memory, z_coord, eps, optimizer, memory_component):
    if optimizer=="heavy_ball":
        return add_state(nominal_memory,z_coord,alpha=eps)
    m0,v0=split_memory(nominal_memory,boundary_model,optimizer)
    zm,zlog=split_memory(z_coord,boundary_model,optimizer)
    if memory_component=="m":
        return add_state(m0,zm,alpha=eps)+clone_state(v0)
    if memory_component=="log_v":
        return clone_state(m0)+[vb*torch.exp(torch.clamp(eps*zl,min=-20,max=20)) for vb,zl in zip(v0,zlog)]
    if memory_component=="joint":
        return add_state(m0,zm,alpha=eps)+[vb*torch.exp(torch.clamp(eps*zl,min=-20,max=20)) for vb,zl in zip(v0,zlog)]
    raise ValueError(memory_component)

def component_radius_base(boundary_model, inherited, optimizer, memory_component):
    if optimizer=="heavy_ball":
        return max(float(norm_state(inherited)),1e-8)
    if memory_component=="m":
        m0,_=adam_memory_parts(inherited,boundary_model)
        return max(float(norm_state(m0)),1e-8)
    return 1.0

def memory_is_nonnegative_v(memory, model, optimizer):
    if optimizer not in ("adam","adamw"):
        return True
    _,v=split_memory(memory,model,optimizer)
    return all(bool(torch.isfinite(x).all() and (x>=0).all()) for x in v)


In [ ]:
def hvp(model,x,y,v):
    _,g=loss_and_grads(model,x,y,create_graph=True)
    gv=dot_state(g,v)
    hv=torch.autograd.grad(gv,params_list(model),retain_graph=False)
    return [h.detach() for h in hv]

_HVP_FALLBACK_WARNED = False
_BATCHED_HVP_ACTIVE = USE_BATCHED_HVP

def hvp_many(model,x,y,vs):
    global _HVP_FALLBACK_WARNED, _BATCHED_HVP_ACTIVE
    if _BATCHED_HVP_ACTIVE and len(vs)>1:
        try:
            params=params_list(model)
            _,g=loss_and_grads(model,x,y,create_graph=True)
            grad_outputs=tuple(torch.stack([v[i] for v in vs],0) for i in range(len(params)))
            hv=torch.autograd.grad(tuple(g),params,grad_outputs=grad_outputs,
                                   retain_graph=False,is_grads_batched=True)
            return [[h[j].detach() for h in hv] for j in range(len(vs))]
        except Exception as e:
            if not _HVP_FALLBACK_WARNED:
                print(f"batched HVP failed once; falling back to per-direction HVPs: {type(e).__name__}: {e}")
                _HVP_FALLBACK_WARNED = True
            _BATCHED_HVP_ACTIVE = False
    return [hvp(model,x,y,v) for v in vs]

def tangent_response_time_varying(boundary_model,batches,B,lr,mu,optimizer="heavy_ball",beta2=np.nan,nominal_memory=None,step_offset=0):
    snaps,times=tangent_response_time_varying_snapshots(boundary_model,batches,B,lr,mu,[len(batches)],optimizer,beta2,nominal_memory,step_offset)
    return snaps[len(batches)], times[len(batches)]

def tangent_response_time_varying_snapshots(boundary_model,batches,B,lr,mu,horizons,optimizer="heavy_ball",beta2=np.nan,nominal_memory=None,step_offset=0):
    wanted=set(int(h) for h in horizons); snapshots={}; timings={}
    model=copy.deepcopy(boundary_model).to(DEVICE)
    memory=clone_state(nominal_memory) if nominal_memory is not None else memory_zeros_like(model,optimizer)
    dtheta=[[torch.zeros_like(p) for p in params_list(model)] for _ in B]
    dmem=[clone_state(b) for b in B]
    tic=time.perf_counter()
    for local_step,(x,y) in enumerate(batches, start=1):
        step=int(step_offset)+local_step
        params=params_list(model)
        if optimizer=="heavy_ball":
            hvs=hvp_many(model,x,y,dtheta)
            for j in range(len(B)):
                dmem[j]=add_state(scale_state(dmem[j],mu),hvs[j])
                dtheta[j]=add_state(dtheta[j],dmem[j],alpha=-lr)
            optimizer_step_(model,memory,x,y,optimizer,lr,mu,beta2,step,CFG.weight_decay)
        elif optimizer in ("adam","adamw"):
            _,g=loss_and_grads(model,x,y,create_graph=True)
            hvs=hvp_many(model,x,y,dtheta)
            g=[gi.detach() for gi in g]
            b1=float(mu); b2=float(beta2); eps=CFG.adam_eps
            c1=adam_bias_correction(step,b1); c2=adam_bias_correction(step,b2)
            m,v=split_memory(memory,model,optimizer)
            g_eff=[gi + CFG.weight_decay*p.detach() if optimizer=="adam" and CFG.weight_decay else gi for gi,p in zip(g,params)]
            h_eff=[add_state(hv,dt,alpha=CFG.weight_decay) if optimizer=="adam" and CFG.weight_decay else hv for hv,dt in zip(hvs,dtheta)]
            m_new=[b1*mi+(1-b1)*gi for mi,gi in zip(m,g_eff)]
            v_new=[b2*vi+(1-b2)*(gi*gi) for vi,gi in zip(v,g_eff)]
            denom=[torch.sqrt(torch.clamp(vi/c2,min=eps*eps)).add(eps) for vi in v_new]
            sqrt_v=[torch.sqrt(torch.clamp(vi/c2,min=eps*eps)) for vi in v_new]
            for j in range(len(B)):
                dm,dv=split_memory(dmem[j],model,optimizer)
                dm_new=[b1*dmi+(1-b1)*hi for dmi,hi in zip(dm,h_eff[j])]
                dv_new=[b2*dvi+2*(1-b2)*gi*hi for dvi,gi,hi in zip(dv,g_eff,h_eff[j])]
                dmhat=[u/c1 for u in dm_new]; dvhat=[u/c2 for u in dv_new]; mhat=[u/c1 for u in m_new]
                du=[dmh/den - mh*(0.5*dvh/torch.clamp(sv,min=eps))/(den*den) for dmh,dvh,mh,den,sv in zip(dmhat,dvhat,mhat,denom,sqrt_v)]
                if optimizer=="adamw" and CFG.weight_decay:
                    dtheta[j]=[dt*(1-lr*CFG.weight_decay)-lr*dui for dt,dui in zip(dtheta[j],du)]
                else:
                    dtheta[j]=[dt-lr*dui for dt,dui in zip(dtheta[j],du)]
                dmem[j]=dm_new+dv_new
            with torch.no_grad():
                for p,mi,vi,den in zip(params,m_new,v_new,denom):
                    if optimizer=="adamw" and CFG.weight_decay: p.mul_(1-lr*CFG.weight_decay)
                    p.addcdiv_(mi/c1,den,value=-lr)
                for old,newv in zip(m,m_new): old.copy_(newv)
                for old,newv in zip(v,v_new): old.copy_(newv)
        else:
            raise ValueError(optimizer)
        if local_step in wanted:
            snapshots[local_step]=[[u.detach().clone() for u in col] for col in dtheta]
            timings[local_step]=time.perf_counter()-tic
    return snapshots,timings

def nonlinear_endpoint(boundary_model,batches,initial_memory,lr,mu,optimizer="heavy_ball",beta2=np.nan,step_offset=0):
    model=copy.deepcopy(boundary_model).to(DEVICE); memory=clone_state(initial_memory)
    for local_step,(x,y) in enumerate(batches, start=1):
        optimizer_step_(model,memory,x,y,optimizer,lr,mu,beta2,int(step_offset)+local_step,CFG.weight_decay)
    return clone_state(params_list(model)), model

def nonlinear_snapshots(boundary_model,batches,initial_memory,lr,mu,horizons,probe_batches=None,optimizer="heavy_ball",beta2=np.nan,step_offset=0):
    wanted=set(int(h) for h in horizons); theta_by_H={}; logits_by_H={}
    model=copy.deepcopy(boundary_model).to(DEVICE); memory=clone_state(initial_memory)
    for local_step,(x,y) in enumerate(batches, start=1):
        optimizer_step_(model,memory,x,y,optimizer,lr,mu,beta2,int(step_offset)+local_step,CFG.weight_decay)
        if local_step in wanted:
            theta_by_H[local_step]=clone_state(params_list(model))
            if probe_batches is not None:
                logits_by_H[local_step]=logits_on_batches(model,probe_batches)
    return theta_by_H, logits_by_H

def empirical_direction(boundary_model,batches,z,lr,mu,eps,optimizer="heavy_ball",beta2=np.nan,nominal_memory=None,step_offset=0,memory_component="momentum"):
    nominal=clone_state(nominal_memory) if nominal_memory is not None else memory_zeros_like(boundary_model,optimizer)
    th0,_=nonlinear_endpoint(boundary_model,batches,nominal,lr,mu,optimizer,beta2,step_offset)
    thp,_=nonlinear_endpoint(boundary_model,batches,memory_from_coordinate(boundary_model,nominal,z,eps,optimizer,memory_component),lr,mu,optimizer,beta2,step_offset)
    thm,_=nonlinear_endpoint(boundary_model,batches,memory_from_coordinate(boundary_model,nominal,z,-eps,optimizer,memory_component),lr,mu,optimizer,beta2,step_offset)
    central=scale_state(add_state(thp,thm,alpha=-1.0),1/(2*eps))
    second=add_state(add_state(thp,th0,alpha=-2.0),thm)
    return central, second, th0


In [ ]:
def midpoint_state(boundary_model,batches,lr,mu,optimizer="heavy_ball",beta2=np.nan,nominal_memory=None,step_offset=0):
    model=copy.deepcopy(boundary_model).to(DEVICE)
    memory=clone_state(nominal_memory) if nominal_memory is not None else memory_zeros_like(model,optimizer)
    mid=len(batches)//2
    for t,(x,y) in enumerate(batches):
        if t==mid: return copy.deepcopy(model), clone_state(memory), (x,y), int(step_offset)+t+1
        optimizer_step_(model,memory,x,y,optimizer,lr,mu,beta2,int(step_offset)+t+1,CFG.weight_decay)
    return model,clone_state(memory),batches[-1],int(step_offset)+len(batches)

def tangent_response_frozen(boundary_model,batches,B,lr,mu,optimizer="heavy_ball",beta2=np.nan,nominal_memory=None,step_offset=0):
    mid_model,mid_memory,(xmid,ymid),mid_step=midpoint_state(boundary_model,batches,lr,mu,optimizer,beta2,nominal_memory,step_offset)
    dtheta=[[torch.zeros_like(p) for p in params_list(mid_model)] for _ in B]
    dmem=[clone_state(b) for b in B]; tic=time.perf_counter()
    params=params_list(mid_model)
    for _ in batches:
        if optimizer=="heavy_ball":
            hvs=hvp_many(mid_model,xmid,ymid,dtheta)
            for j in range(len(B)):
                dmem[j]=add_state(scale_state(dmem[j],mu),hvs[j])
                dtheta[j]=add_state(dtheta[j],dmem[j],alpha=-lr)
        elif optimizer in ("adam","adamw"):
            _,g=loss_and_grads(mid_model,xmid,ymid,create_graph=True)
            hvs=hvp_many(mid_model,xmid,ymid,dtheta)
            g=[gi.detach() for gi in g]
            b1=float(mu); b2=float(beta2); eps=CFG.adam_eps
            c1=adam_bias_correction(mid_step,b1); c2=adam_bias_correction(mid_step,b2)
            m,v=split_memory(mid_memory,mid_model,optimizer)
            g_eff=[gi + CFG.weight_decay*p.detach() if optimizer=="adam" and CFG.weight_decay else gi for gi,p in zip(g,params)]
            h_eff=[add_state(hv,dt,alpha=CFG.weight_decay) if optimizer=="adam" and CFG.weight_decay else hv for hv,dt in zip(hvs,dtheta)]
            m_new=[b1*mi+(1-b1)*gi for mi,gi in zip(m,g_eff)]
            v_new=[b2*vi+(1-b2)*(gi*gi) for vi,gi in zip(v,g_eff)]
            denom=[torch.sqrt(torch.clamp(vi/c2,min=eps*eps)).add(eps) for vi in v_new]
            sqrt_v=[torch.sqrt(torch.clamp(vi/c2,min=eps*eps)) for vi in v_new]
            for j in range(len(B)):
                dm,dv=split_memory(dmem[j],mid_model,optimizer)
                dm_new=[b1*dmi+(1-b1)*hi for dmi,hi in zip(dm,h_eff[j])]
                dv_new=[b2*dvi+2*(1-b2)*gi*hi for dvi,gi,hi in zip(dv,g_eff,h_eff[j])]
                dmhat=[u/c1 for u in dm_new]; dvhat=[u/c2 for u in dv_new]; mhat=[u/c1 for u in m_new]
                du=[dmh/den - mh*(0.5*dvh/torch.clamp(sv,min=eps))/(den*den) for dmh,dvh,mh,den,sv in zip(dmhat,dvhat,mhat,denom,sqrt_v)]
                if optimizer=="adamw" and CFG.weight_decay:
                    dtheta[j]=[dt*(1-lr*CFG.weight_decay)-lr*dui for dt,dui in zip(dtheta[j],du)]
                else:
                    dtheta[j]=[dt-lr*dui for dt,dui in zip(dtheta[j],du)]
                dmem[j]=dm_new+dv_new
        else:
            raise ValueError(optimizer)
    return dtheta,time.perf_counter()-tic


In [ ]:
AUDIT_EPSILONS=(1e-2,1e-3,1e-4,1e-5,1e-6)
AUDIT_TOLERANCES=dict(
    one_step_jvp=1e-8,
    hvp=1e-8,
    adam_jvp=1e-7,
    dense_jacobian=1e-7,
    reduced_basis=1e-8,
    residual_slope_lo=1.7,
    residual_slope_hi=2.3,
)

def fd_regime_pass(errors, tol):
    vals=[float(e) for e in errors if np.isfinite(e)]
    if not vals:
        return False, np.nan, "no finite centered-difference errors"
    if min(vals) < tol:
        return True, min(vals), f"minimum centered-difference error {min(vals):.3e} is below tolerance {tol:.1e}"
    for a,b in zip(vals, vals[1:]):
        if b < 0.5*a:
            return True, min(vals), "centered-difference error has a decreasing local regime"
    return False, min(vals), "no decreasing regime and minimum error above tolerance"

def residual_slope_pass(residual, pred_norm, tol=1e-10):
    vals=[(float(e),float(r)) for e,r in residual if np.isfinite(r) and r>0]
    if not vals:
        return True, np.nan, "residual is numerically zero"
    scale=max(float(pred_norm),1.0)
    if max(r for _,r in vals) < tol*scale:
        return True, np.nan, f"first-order residual is at numerical floor below {tol:.1e}*scale"
    xs=np.log([e for e,r in vals[:4]]); ys=np.log([r for e,r in vals[:4]])
    slope=float(np.polyfit(xs,ys,1)[0]) if len(xs)>=2 else np.nan
    passed=np.isfinite(slope) and AUDIT_TOLERANCES['residual_slope_lo']<=slope<=AUDIT_TOLERANCES['residual_slope_hi']
    return passed, slope, "slope in accepted O(eps^2) range" if passed else "slope outside accepted O(eps^2) range"

def rel_l2(a,b,tol=1e-12):
    den=float(torch.linalg.vector_norm(b).detach())
    num=float(torch.linalg.vector_norm((a-b)).detach())
    return num/max(den,tol)

def cos_flat(a,b,tol=1e-12):
    na=float(torch.linalg.vector_norm(a).detach()); nb=float(torch.linalg.vector_norm(b).detach())
    if min(na,nb)<tol: return np.nan
    return float(torch.dot(a.reshape(-1),b.reshape(-1))/(torch.linalg.vector_norm(a)*torch.linalg.vector_norm(b)))

def audit_write(name, df):
    write_dataframe(df, TAB/name)
    return df

def audit_tensor(seed, shape, scale=0.2):
    gen=torch.Generator(device='cpu').manual_seed(seed)
    return (scale*torch.randn(*shape, generator=gen, dtype=torch.float64)).to(DEVICE).detach()

def audit_param_shapes():
    return [(3,4),(3,),(2,3),(2,)]

def audit_numel(shapes=None):
    shapes=audit_param_shapes() if shapes is None else shapes
    return int(sum(math.prod(s) for s in shapes))

def audit_unflatten_flat(v, shapes=None):
    shapes=audit_param_shapes() if shapes is None else shapes
    out=[]; i=0
    for shape in shapes:
        n=math.prod(shape); out.append(v[i:i+n].reshape(shape)); i+=n
    assert i==v.numel()
    return out

def audit_tiny_forward(theta_flat, x, shapes=None):
    W1,b1,W2,b2=audit_unflatten_flat(theta_flat, shapes)
    h=torch.tanh(x @ W1.T + b1)
    return h @ W2.T + b2

def audit_tiny_loss(theta_flat, x, y):
    return F.cross_entropy(audit_tiny_forward(theta_flat,x), y)

def audit_grad(theta_flat, x, y, create_graph=True):
    loss=audit_tiny_loss(theta_flat,x,y)
    return torch.autograd.grad(loss, theta_flat, create_graph=create_graph)[0]

def audit_hvp_flat(theta_flat, x, y, u):
    g=audit_grad(theta_flat,x,y,create_graph=True)
    return torch.autograd.grad(torch.dot(g,u), theta_flat, retain_graph=False)[0]

def audit_make_module(theta_flat):
    class TinyAuditNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1=nn.Linear(4,3)
            self.fc2=nn.Linear(3,2)
        def forward(self,x):
            return self.fc2(torch.tanh(self.fc1(x)))
    model=TinyAuditNet().to(DEVICE).double()
    parts=audit_unflatten_flat(theta_flat)
    with torch.no_grad():
        model.fc1.weight.copy_(parts[0]); model.fc1.bias.copy_(parts[1])
        model.fc2.weight.copy_(parts[2]); model.fc2.bias.copy_(parts[3])
    return model

def audit_flatten_module_params(model):
    return flatten_state(params_list(model))

def audit_equations_table():
    rows=[
        dict(optimizer='heavy_ball',equation='g_eff = grad L(theta) + wd * theta if wd>0 else grad L(theta); m_next = mu*m + g_eff; theta_next = theta - lr*m_next',
             dampening='none',nesterov='false',weight_decay='coupled L2',bias_correction='none',amsgrad='false',maximize='false',scheduler='none',parameter_groups='single implicit group'),
        dict(optimizer='adam',equation='g_eff = grad L(theta) + wd * theta if wd>0 else grad L(theta); m_next = beta1*m + (1-beta1)*g_eff; v_next = beta2*v + (1-beta2)*g_eff^2; theta_next = theta - lr*(m_next/c1)/(sqrt(clamp(v_next/c2, eps^2))+eps)',
             dampening='not applicable',nesterov='false',weight_decay='coupled L2',bias_correction='c1=1-beta1^step, c2=1-beta2^step',amsgrad='false',maximize='false',scheduler='none',parameter_groups='single implicit group'),
        dict(optimizer='adamw',equation='g = grad L(theta); m_next = beta1*m + (1-beta1)*g; v_next = beta2*v + (1-beta2)*g^2; theta_next = (1-lr*wd)*theta - lr*(m_next/c1)/(sqrt(clamp(v_next/c2, eps^2))+eps)',
             dampening='not applicable',nesterov='false',weight_decay='decoupled AdamW',bias_correction='c1=1-beta1^step, c2=1-beta2^step',amsgrad='false',maximize='false',scheduler='none',parameter_groups='single implicit group'),
    ]
    return audit_write('audit_optimizer_equations', pd.DataFrame(rows))

def audit_packing_test(theta,x,y):
    rows=[]
    refs={
        'parameters': audit_unflatten_flat(theta),
        'gradients': audit_unflatten_flat(audit_grad(theta.clone().requires_grad_(True),x,y,create_graph=False)),
        'hvp': audit_unflatten_flat(audit_hvp_flat(theta.clone().requires_grad_(True),x,y,audit_tensor(101,(theta.numel(),)))),
        'momentum': audit_unflatten_flat(audit_tensor(102,(theta.numel(),))),
        'adam_m': audit_unflatten_flat(audit_tensor(103,(theta.numel(),))),
        'adam_v': audit_unflatten_flat(torch.rand(theta.numel(), dtype=torch.float64, device=DEVICE)+0.1),
        'tangent': audit_unflatten_flat(audit_tensor(104,(theta.numel(),))),
        'response': audit_unflatten_flat(audit_tensor(105,(theta.numel(),))),
    }
    start=0
    slices=[]
    for i,shape in enumerate(audit_param_shapes()):
        n=math.prod(shape); slices.append(dict(param_index=i,start=start,stop=start+n,shape=str(shape))); start+=n
    for name,state in refs.items():
        flat=flatten_state(state); rt=unflatten_like(flat,state); flat2=flatten_state(rt)
        rows.append(dict(object=name,numel=int(flat.numel()),roundtrip_rel_error=rel_l2(flat2,flat),passed=rel_l2(flat2,flat)<1e-12, slices=json.dumps(slices)))
    return audit_write('audit_packing', pd.DataFrame(rows))

def audit_hvp_test(theta,x,y):
    theta_req=theta.clone().detach().requires_grad_(True)
    u=audit_tensor(201,(theta.numel(),))
    H=torch.autograd.functional.hessian(lambda th: audit_tiny_loss(th,x,y), theta_req)
    hv_dense=H @ u
    hv_direct=audit_hvp_flat(theta_req,x,y,u).detach()
    def grad_fn(th): return audit_grad(th,x,y,create_graph=True)
    _,hv_jvp=torch.autograd.functional.jvp(grad_fn, (theta_req,), (u,), create_graph=False)
    model=audit_make_module(theta)
    hv_module=flatten_state(hvp(model,x,y,audit_unflatten_flat(u))).double()
    rows=[]
    for name,val in [('dense_vs_direct',hv_dense),('jvp_vs_direct',hv_jvp),('module_hvp_vs_direct',hv_module)]:
        err=rel_l2(val.detach(),hv_direct); rows.append(dict(check=name,relative_error=err,cosine=cos_flat(val.detach(),hv_direct),passed=err<AUDIT_TOLERANCES['hvp']))
    return audit_write('audit_hvp', pd.DataFrame(rows))

def sgd_step_flat(theta,m,x,y,lr,mu,wd):
    theta_var=theta if theta.requires_grad else theta.clone().detach().requires_grad_(True)
    g=audit_grad(theta_var,x,y,create_graph=True)
    geff=g+wd*theta_var if wd else g
    m_next=mu*m+geff
    theta_next=theta_var-lr*m_next
    return theta_next,m_next

def sgd_state_map(state,x,y,lr,mu,wd):
    n=state.numel()//2; theta=state[:n]; m=state[n:]
    th,mn=sgd_step_flat(theta,m,x,y,lr,mu,wd)
    return torch.cat([th,mn])

def sgd_manual_tangent(theta,dtheta,dm,x,y,lr,mu,wd):
    hv=audit_hvp_flat(theta.clone().detach().requires_grad_(True),x,y,dtheta)
    if wd: hv=hv+wd*dtheta
    dm_next=mu*dm+hv
    dtheta_next=dtheta-lr*dm_next
    return dtheta_next.detach(),dm_next.detach()

def audit_one_step_sgd_test(theta,x,y):
    lr=0.07; mu=0.63; wd=0.013
    m=audit_tensor(301,(theta.numel(),)); dtheta=audit_tensor(302,(theta.numel(),)); dm=audit_tensor(303,(theta.numel(),))
    state=torch.cat([theta,m]).detach().requires_grad_(True); dstate=torch.cat([dtheta,dm])
    manual=torch.cat(sgd_manual_tangent(theta,dtheta,dm,x,y,lr,mu,wd))
    _,jvp_val=torch.autograd.functional.jvp(lambda s: sgd_state_map(s,x,y,lr,mu,wd), (state,), (dstate,), create_graph=False)
    rows=[dict(epsilon=np.nan,comparison='manual_vs_jvp',relative_error=rel_l2(manual,jvp_val.detach()),cosine=cos_flat(manual,jvp_val.detach()),passed=rel_l2(manual,jvp_val.detach())<AUDIT_TOLERANCES['one_step_jvp'])]
    fd_errors=[]
    for eps in AUDIT_EPSILONS:
        fd=(sgd_state_map((state.detach()+eps*dstate).requires_grad_(True),x,y,lr,mu,wd)-sgd_state_map((state.detach()-eps*dstate).requires_grad_(True),x,y,lr,mu,wd))/(2*eps)
        err=rel_l2(manual,fd.detach()); fd_errors.append(err)
        rows.append(dict(epsilon=eps,comparison='manual_vs_centered_fd',relative_error=err,cosine=cos_flat(manual,fd.detach()),passed=np.isfinite(err)))
    fd_pass,fd_min,fd_detail=fd_regime_pass(fd_errors, AUDIT_TOLERANCES['one_step_jvp'])
    rows.append(dict(epsilon=np.nan,comparison='centered_fd_has_decreasing_or_roundoff_regime',relative_error=fd_min,cosine=np.nan,
                     passed=fd_pass,detail=fd_detail))
    return audit_write('audit_one_step_sgd', pd.DataFrame(rows))

def sgd_multistep(theta,m,batches,lr,mu,wd,H):
    th=theta
    mm=m
    for x,y in batches[:H]: th,mm=sgd_step_flat(th,mm,x,y,lr,mu,wd)
    return th,mm

def sgd_multistep_tangent(theta,m,z,batches,lr,mu,wd,H):
    th=theta.clone().detach(); dtheta=torch.zeros_like(theta); dm=z.clone()
    mm=m.clone().detach()
    for x,y in batches[:H]:
        hv=audit_hvp_flat(th.clone().detach().requires_grad_(True),x,y,dtheta)
        if wd: hv=hv+wd*dtheta
        g=audit_grad(th.clone().detach().requires_grad_(True),x,y,create_graph=False).detach()
        geff=g+wd*th if wd else g
        dm=mu*dm+hv.detach(); dtheta=dtheta-lr*dm
        mm=mu*mm+geff; th=th-lr*mm
    return dtheta.detach()

def audit_multistep_sgd_test(theta,batches):
    lr=0.04; mu=0.77; wd=0.0; m0=audit_tensor(401,(theta.numel(),)); z=audit_tensor(402,(theta.numel(),))
    rows=[]
    for H in (1,2,5,10,20):
        pred=sgd_multistep_tangent(theta,m0,z,batches,lr,mu,wd,H)
        th0,_=sgd_multistep(theta,m0,batches,lr,mu,wd,H)
        residual=[]; fd_errors=[]
        for eps in AUDIT_EPSILONS:
            thp,_=sgd_multistep(theta,m0+eps*z,batches,lr,mu,wd,H)
            thm,_=sgd_multistep(theta,m0-eps*z,batches,lr,mu,wd,H)
            fd=(thp-thm)/(2*eps)
            disp=thp-th0
            r=float(torch.linalg.vector_norm((disp-eps*pred)).detach())
            err=rel_l2(pred,fd.detach()); fd_errors.append(err)
            residual.append((eps,r))
            rows.append(dict(horizon=H,epsilon=eps,comparison='tangent_vs_centered_fd',relative_error=err,cosine=cos_flat(pred,fd.detach()),first_order_residual=r,passed=np.isfinite(err)))
        fd_pass,fd_min,fd_detail=fd_regime_pass(fd_errors, 1e-7)
        rows.append(dict(horizon=H,epsilon=np.nan,comparison='centered_fd_has_decreasing_or_roundoff_regime',relative_error=fd_min,cosine=np.nan,first_order_residual=np.nan,
                         passed=fd_pass,detail=fd_detail))
        slope_pass,slope,slope_detail=residual_slope_pass(residual,float(torch.linalg.vector_norm(pred).detach()))
        rows.append(dict(horizon=H,epsilon=np.nan,comparison='first_order_residual_slope_or_roundoff_floor',relative_error=np.nan,cosine=np.nan,first_order_residual=slope,passed=slope_pass,detail=slope_detail))
    return audit_write('audit_multistep_sgd', pd.DataFrame(rows))

def audit_time_ordering_test():
    M0=torch.tensor([[1.0,2.0],[0.0,1.0]],dtype=torch.float64,device=DEVICE)
    M1=torch.tensor([[1.0,0.0],[3.0,1.0]],dtype=torch.float64,device=DEVICE)
    z=torch.tensor([0.4,-0.7],dtype=torch.float64,device=DEVICE)
    state=z.clone()
    for M in [M0,M1]: state=M@state
    chronological=M1@(M0@z); reversed_product=M0@(M1@z)
    err=rel_l2(state,chronological); reverse_gap=rel_l2(reversed_product,chronological)
    df=pd.DataFrame([dict(check='chronological_product_M1_M0',relative_error=err,reversed_gap=reverse_gap,passed=err<1e-12 and reverse_gap>1e-2)])
    return audit_write('audit_time_ordering', df)

def audit_dense_jacobian_test(theta,batches):
    lr=0.04; mu=0.77; wd=0.0; H=5; m0=audit_tensor(501,(theta.numel(),))
    def theta_from_m(m): return sgd_multistep(theta,m,batches,lr,mu,wd,H)[0]
    J=torch.autograd.functional.jacobian(theta_from_m, m0)
    eye=torch.eye(theta.numel(),dtype=torch.float64,device=DEVICE)
    cols=torch.stack([sgd_multistep_tangent(theta,m0,eye[i],batches,lr,mu,wd,H) for i in range(theta.numel())],dim=1)
    matrix_err=rel_l2(cols,J.detach())
    rows=[dict(check='dense_jacobian_columns',relative_error=matrix_err,cosine=np.nan,passed=matrix_err<AUDIT_TOLERANCES['dense_jacobian'])]
    for k in range(5):
        z=audit_tensor(510+k,(theta.numel(),))
        pred=sgd_multistep_tangent(theta,m0,z,batches,lr,mu,wd,H)
        dense=J.detach()@z
        err=rel_l2(pred,dense); rows.append(dict(check=f'dense_jacobian_random_z_{k}',relative_error=err,cosine=cos_flat(pred,dense),passed=err<AUDIT_TOLERANCES['dense_jacobian']))
    return audit_write('audit_dense_jacobian', pd.DataFrame(rows))

def adam_step_flat(theta,m,v,x,y,lr,b1,b2,eps,step,wd,optimizer):
    theta_var=theta if theta.requires_grad else theta.clone().detach().requires_grad_(True)
    g=audit_grad(theta_var,x,y,create_graph=True)
    geff=g+wd*theta_var if optimizer=='adam' and wd else g
    m_next=b1*m+(1-b1)*geff
    v_next=b2*v+(1-b2)*(geff*geff)
    c1=adam_bias_correction(step,b1); c2=adam_bias_correction(step,b2)
    mhat=m_next/c1; vhat=v_next/c2
    denom=torch.sqrt(torch.clamp(vhat,min=eps*eps))+eps
    theta_base=theta_var*(1-lr*wd) if optimizer=='adamw' and wd else theta_var
    theta_next=theta_base-lr*mhat/denom
    return theta_next,m_next,v_next

def adam_manual_tangent(theta,m,v,dtheta,dm,dv,x,y,lr,b1,b2,eps,step,wd,optimizer):
    hv=audit_hvp_flat(theta.clone().detach().requires_grad_(True),x,y,dtheta)
    g=audit_grad(theta.clone().detach().requires_grad_(True),x,y,create_graph=False).detach()
    if optimizer=='adam' and wd:
        geff=g+wd*theta; h_eff=hv+wd*dtheta
    else:
        geff=g; h_eff=hv
    m_next=b1*m+(1-b1)*geff
    v_next=b2*v+(1-b2)*(geff*geff)
    dm_next=b1*dm+(1-b1)*h_eff
    dv_next=b2*dv+2*(1-b2)*geff*h_eff
    c1=adam_bias_correction(step,b1); c2=adam_bias_correction(step,b2)
    mhat=m_next/c1; vhat=v_next/c2; dmhat=dm_next/c1; dvhat=dv_next/c2
    sqrt_v=torch.sqrt(torch.clamp(vhat,min=eps*eps)); denom=sqrt_v+eps
    du=dmhat/denom - mhat*(0.5*dvhat/torch.clamp(sqrt_v,min=eps))/(denom*denom)
    dtheta_next=dtheta*(1-lr*wd)-lr*du if optimizer=='adamw' and wd else dtheta-lr*du
    return dtheta_next.detach(),dm_next.detach(),dv_next.detach()

def adam_state_from_coord(theta,m,v,coord,eps_scale,component):
    n=theta.numel()
    if component=='m':
        dm=coord; return theta,m+eps_scale*dm,v
    if component=='v_additive':
        dv=coord; return theta,m,v+eps_scale*dv
    if component=='log_v':
        zlog=coord; return theta,m,v*torch.exp(torch.clamp(eps_scale*zlog,min=-20,max=20))
    if component=='joint':
        dm=coord[:n]; zlog=coord[n:]; return theta,m+eps_scale*dm,v*torch.exp(torch.clamp(eps_scale*zlog,min=-20,max=20))
    raise ValueError(component)

def adam_coordinate_tangent(theta,m,v,coord,component):
    n=theta.numel(); dtheta=torch.zeros_like(theta)
    if component=='m': return dtheta,coord,torch.zeros_like(v)
    if component=='v_additive': return dtheta,torch.zeros_like(m),coord
    if component=='log_v': return dtheta,torch.zeros_like(m),v*coord
    if component=='joint': return dtheta,coord[:n],v*coord[n:]
    raise ValueError(component)

def audit_adam_one_step_test(theta,x,y):
    lr=0.03; b1=0.82; b2=0.91; eps=1e-6; step=17
    rows=[]; n=theta.numel(); m=audit_tensor(601,(n,)); v=torch.rand(n,dtype=torch.float64,device=DEVICE)+0.2
    for optimizer,wd in [('adam',0.011),('adamw',0.013)]:
        for component in ['m','v_additive','log_v','joint']:
            coord=audit_tensor(610+len(rows),(n if component!='joint' else 2*n,))
            dtheta,dm,dv=adam_coordinate_tangent(theta,m,v,coord,component)
            manual=torch.cat(adam_manual_tangent(theta,m,v,dtheta,dm,dv,x,y,lr,b1,b2,eps,step,wd,optimizer))
            def map_coord(c):
                th0,m0,v0=adam_state_from_coord(theta,m,v,c,1.0,component)
                return torch.cat(adam_step_flat(th0,m0,v0,x,y,lr,b1,b2,eps,step,wd,optimizer))
            z0=torch.zeros_like(coord).requires_grad_(True)
            _,jvp_val=torch.autograd.functional.jvp(map_coord,(z0,),(coord,),create_graph=False)
            jerr=rel_l2(manual,jvp_val.detach())
            rows.append(dict(optimizer=optimizer,memory_component=component,epsilon=np.nan,comparison='manual_vs_jvp',relative_error=jerr,cosine=cos_flat(manual,jvp_val.detach()),v_nonnegative=bool((v>0).all()),passed=jerr<AUDIT_TOLERANCES['adam_jvp']))
            fd_errors=[]
            for fd_eps in AUDIT_EPSILONS:
                plus=torch.cat(adam_step_flat(*adam_state_from_coord(theta,m,v,coord,fd_eps,component),x,y,lr,b1,b2,eps,step,wd,optimizer))
                minus=torch.cat(adam_step_flat(*adam_state_from_coord(theta,m,v,coord,-fd_eps,component),x,y,lr,b1,b2,eps,step,wd,optimizer))
                fd=(plus-minus)/(2*fd_eps)
                err=rel_l2(manual,fd.detach()); fd_errors.append(err)
                rows.append(dict(optimizer=optimizer,memory_component=component,epsilon=fd_eps,comparison='manual_vs_centered_fd',relative_error=err,cosine=cos_flat(manual,fd.detach()),v_nonnegative=bool((adam_state_from_coord(theta,m,v,coord,fd_eps,component)[2]>=0).all() and (adam_state_from_coord(theta,m,v,coord,-fd_eps,component)[2]>=0).all()),passed=np.isfinite(err)))
            fd_pass,fd_min,fd_detail=fd_regime_pass(fd_errors, AUDIT_TOLERANCES['adam_jvp'])
            rows.append(dict(optimizer=optimizer,memory_component=component,epsilon=np.nan,comparison='centered_fd_has_decreasing_or_roundoff_regime',relative_error=fd_min,cosine=np.nan,v_nonnegative=True,
                             passed=fd_pass,detail=fd_detail))
    
    dtheta=audit_tensor(777,(n,)); dm=audit_tensor(778,(n,)); dv=audit_tensor(779,(n,))
    correct=torch.cat(adam_manual_tangent(theta,m,v,dtheta,dm,dv,x,y,lr,b1,b2,eps,step,0.02,'adamw'))
    wrong_theta,wrong_m,wrong_v=adam_manual_tangent(theta,m,v,dtheta,dm,dv,x,y,lr,b1,b2,eps,step,0.0,'adamw')
    wrong=torch.cat([wrong_theta,wrong_m,wrong_v])
    rows.append(dict(optimizer='adamw',memory_component='decay_guard',epsilon=np.nan,comparison='decoupled_decay_differs_from_no_decay',relative_error=rel_l2(wrong,correct),cosine=cos_flat(wrong,correct),v_nonnegative=True,passed=rel_l2(wrong,correct)>1e-5))
    coupled=torch.cat(adam_manual_tangent(theta,m,v,dtheta,dm,dv,x,y,lr,b1,b2,eps,step,0.02,'adam'))
    rows.append(dict(optimizer='adamw',memory_component='decay_guard',epsilon=np.nan,comparison='decoupled_adamw_differs_from_coupled_l2_adam',relative_error=rel_l2(coupled,correct),cosine=cos_flat(coupled,correct),v_nonnegative=True,passed=rel_l2(coupled,correct)>1e-5))
    return audit_write('audit_adam_one_step', pd.DataFrame(rows))

def audit_reduced_basis_test(theta,batches):
    lr=0.04; mu=0.77; wd=0.0; H=5; m0=audit_tensor(801,(theta.numel(),))
    B=[audit_tensor(810+i,(theta.numel(),)) for i in range(4)]
    cols=[sgd_multistep_tangent(theta,m0,b,batches,lr,mu,wd,H) for b in B]
    alpha=np.array([0.2,-0.7,0.4,0.1])
    z=sum(float(a)*b for a,b in zip(alpha,B))
    direct=sgd_multistep_tangent(theta,m0,z,batches,lr,mu,wd,H)
    combined=sum(float(a)*c for a,c in zip(alpha,cols))
    lin_err=rel_l2(direct,combined)

    model=audit_make_module(theta)
    B_state=[audit_unflatten_flat(b) for b in B]
    batched,_=tangent_response_time_varying_snapshots(model,batches[:H],B_state,lr,mu,[H],"heavy_ball",np.nan,memory_zeros_like(model,"heavy_ball"),0)
    independent=[]
    for b in B_state:
        single,_=tangent_response_time_varying_snapshots(model,batches[:H],[b],lr,mu,[H],"heavy_ball",np.nan,memory_zeros_like(model,"heavy_ball"),0)
        independent.append(single[H][0])
    batched_flat=[flatten_state(c).double() for c in batched[H]]
    independent_flat=[flatten_state(c).double() for c in independent]
    block_err=max(rel_l2(a,b) for a,b in zip(batched_flat,independent_flat))
    block_combined=sum(float(a)*c for a,c in zip(alpha,batched_flat))
    block_direct=flatten_state(tangent_response_time_varying_snapshots(model,batches[:H],[audit_unflatten_flat(z)],lr,mu,[H],"heavy_ball",np.nan,memory_zeros_like(model,"heavy_ball"),0)[0][H][0]).double()
    block_linear_err=rel_l2(block_direct,block_combined)
    rows=[dict(check='flat_R_B_alpha_equals_RB_alpha',relative_error=lin_err,passed=lin_err<AUDIT_TOLERANCES['reduced_basis']),
          dict(check='notebook_batched_columns_equal_independent',relative_error=block_err,passed=block_err<AUDIT_TOLERANCES['reduced_basis']),
          dict(check='notebook_R_B_alpha_equals_RB_alpha',relative_error=block_linear_err,passed=block_linear_err<AUDIT_TOLERANCES['reduced_basis'])]
    return audit_write('audit_reduced_basis', pd.DataFrame(rows))

def audit_state_inventory_table():
    rows=[
        dict(state_variable='parameters theta',smallcnn='propagated and differentiated',resnet18='propagated and differentiated',audit_semantics='included in x_t'),
        dict(state_variable='heavy-ball momentum m',smallcnn='perturbed for momentum response; propagated and differentiated',resnet18='same',audit_semantics='optimizer state'),
        dict(state_variable='Adam first moment m',smallcnn='perturbed in m and joint channels; propagated and differentiated',resnet18='same',audit_semantics='optimizer state'),
        dict(state_variable='Adam second moment v',smallcnn='perturbed through additive-v audit and log-v experiment; propagated and differentiated',resnet18='same',audit_semantics='optimizer state'),
        dict(state_variable='Adam step counter tau',smallcnn='held fixed at boundary offset; incremented deterministically; not perturbed',resnet18='same',audit_semantics='discrete optimizer state for bias correction'),
        dict(state_variable='BatchNorm buffers',smallcnn='none',resnet18='present; current continuation uses train-mode modules, so buffers can propagate but are not differentiated by theta-only RTTP',audit_semantics='flagged interpretation issue unless frozen or included'),
        dict(state_variable='scheduler state',smallcnn='none',resnet18='none',audit_semantics='held absent'),
        dict(state_variable='AMP gradient scaler',smallcnn=f'absent for response; boundary use_amp_boundary={CFG.use_amp_boundary}',resnet18=f'absent for response; boundary use_amp_boundary={CFG.use_amp_boundary}',audit_semantics='not part of response state'),
        dict(state_variable='minibatches/augmentations',smallcnn='materialized before response; held fixed',resnet18='materialized before response; held fixed',audit_semantics='external randomness fixed'),
        dict(state_variable='dropout masks',smallcnn='none',resnet18='none in torchvision ResNet-18',audit_semantics='absent'),
    ]
    return audit_write('audit_state_inventory', pd.DataFrame(rows))

def audit_batchnorm_semantics():
    rows=[]
    for arch in CFG.architectures:
        model=build_model(arch)
        bn=[m for m in model.modules() if isinstance(m, nn.modules.batchnorm._BatchNorm)]
        rows.append(dict(architecture=arch,batchnorm_layers=len(bn),has_batchnorm=len(bn)>0,
                         continuation_mode='train',buffers_differentiated=False,
                         interpretation='OK: no BatchNorm buffers' if not bn else 'WARNING: BatchNorm buffers are propagated by train-mode forward passes but omitted from theta-only tangent state'))
    return pd.DataFrame(rows)

def audit_report_tables(tables):
    report=[]
    def add(section,check,passed,detail,blocking=True):
        report.append(dict(section=section,check=check,passed=bool(passed),blocking=bool(blocking),detail=str(detail)))
    for name,df in tables.items():
        if 'passed' in df.columns:
            failed=df[~df.passed.astype(bool)]
            detail_df=failed if len(failed) else df
            add('Numerical identities',name,bool(df.passed.all()),detail_df.to_dict('records')[:8],True)
    bn=audit_batchnorm_semantics()
    for _,r in bn.iterrows():
        add('State semantics',f'batchnorm_{r.architecture}',not bool(r.has_batchnorm),r.interpretation,bool(r.has_batchnorm))
    add('Exact equations','optimizer_equations_documented',True,'audit_optimizer_equations.csv contains exact code equations',False)
    return audit_write('audit_report', pd.DataFrame(report))

def run_finite_horizon_response_audit():
    if not CFG.run_audit:
        print('finite-horizon audit skipped because CFG.run_audit=False')
        return pd.DataFrame()
    old_dtype=torch.get_default_dtype()
    torch.set_default_dtype(torch.float64)
    try:
        seed_everything(12345)
        n=audit_numel(); theta=audit_tensor(11,(n,),scale=0.15).requires_grad_(True)
        x=audit_tensor(12,(7,4),scale=0.7); y=torch.tensor([0,1,0,1,1,0,1],device=DEVICE,dtype=torch.long)
        batches=[]
        for k in range(20):
            xb=audit_tensor(20+k,(6,4),scale=0.6)
            yb=torch.tensor([(i+k)%2 for i in range(6)],device=DEVICE,dtype=torch.long)
            batches.append((xb,yb))
        tables={}
        tables['audit_optimizer_equations']=audit_equations_table()
        tables['audit_packing']=audit_packing_test(theta.detach(),x,y)
        tables['audit_hvp']=audit_hvp_test(theta.detach(),x,y)
        tables['audit_one_step_sgd']=audit_one_step_sgd_test(theta.detach(),x,y)
        tables['audit_multistep_sgd']=audit_multistep_sgd_test(theta.detach(),batches)
        tables['audit_time_ordering']=audit_time_ordering_test()
        tables['audit_dense_jacobian']=audit_dense_jacobian_test(theta.detach(),batches)
        tables['audit_adam_one_step']=audit_adam_one_step_test(theta.detach(),x,y)
        tables['audit_reduced_basis']=audit_reduced_basis_test(theta.detach(),batches)
        tables['audit_state_inventory']=audit_state_inventory_table()
        report=audit_report_tables(tables)
        failed_numerical=report[(report.section=="Numerical identities")&(report.blocking)&(~report.passed)]
        failed_interpretation=report[(report.section=="State semantics")&(report.blocking)&(~report.passed)]
        display(report)
        if len(failed_interpretation):
            print("finite-horizon audit warning: state-semantics issues block paper-scale interpretation until addressed:", failed_interpretation[["check","detail"]].to_dict("records"))
        if len(failed_numerical):
            raise AssertionError(f'finite-horizon numerical audit failed: {failed_numerical[["check","detail"]].to_dict("records")}')
        return report
    finally:
        torch.set_default_dtype(old_dtype)

finite_horizon_audit_report=run_finite_horizon_response_audit()
finite_horizon_audit_report


In [ ]:
def state_rel_error(pred,true,tol=1e-10):
    nt=float(norm_state(true))
    if nt<tol: return np.nan
    return float(norm_state(add_state(pred,true,alpha=-1.0))/norm_state(true))
def state_cos(a,b,tol=1e-10):
    na=float(norm_state(a)); nb=float(norm_state(b))
    if min(na,nb)<tol: return np.nan
    return float(dot_state(a,b)/(norm_state(a)*norm_state(b)))
def state_norm_float(a): return float(norm_state(a))

def safe_ratio(num,den,tol=1e-12):
    return float(num/den) if np.isfinite(num) and np.isfinite(den) and abs(den)>tol else np.nan

@torch.no_grad()
def logits_on_batches(model,batches): return torch.cat([model(x).detach().cpu() for x,_ in batches],0)

REQUIRED_RESPONSE_COLUMNS={
    "prediction_norm","frozen_prediction_norm","actual_gain","response_gain","frozen_response_gain",
    "frozen_modeling_gap","magnitude_error","angular_error","central_magnitude_error","central_angular_error",
    "memory_component","state_parameterization","nominal_memory","step_offset"
}

REQUIRED_SETTING_COLUMNS={"optimizer","memory_component","state_parameterization","nominal_memory","step_offset","beta2"}

def columns_frobenius_norm(S):
    return math.sqrt(max(sum(float(dot_state(s,s)) for s in S),0.0))

def response_spectrum_singular_values_from_gram(G):
    G=((G+G.T)/2).to(dtype=torch.float64)
    G=torch.nan_to_num(G, nan=0.0, posinf=1e300, neginf=-1e300)
    scale=float(torch.linalg.vector_norm(G).detach())
    if not np.isfinite(scale) or scale==0.0:
        return np.zeros(G.shape[0], dtype=float), "zero_or_nonfinite_gram"
    Gs=G/scale
    eye=torch.eye(Gs.shape[0],dtype=Gs.dtype,device=Gs.device)
    last_error=None
    for jitter in (0.0,1e-14,1e-12,1e-10,1e-8):
        try:
            eig=torch.linalg.eigvalsh(Gs + jitter*eye).clamp_min(0)
            sing=torch.sqrt(eig*scale).flip(0).cpu().numpy()
            return sing, ("eigvalsh" if jitter==0.0 else f"eigvalsh_jitter_{jitter:g}")
        except Exception as e:
            last_error=e
    try:
        _,Svals,_=torch.linalg.svd(Gs)
        sing=torch.sqrt(torch.clamp(Svals*scale,min=0)).cpu().numpy()
        return sing, "svd_fallback"
    except Exception as e:
        print(f"spectrum fallback failed; returning zeros. eig error={last_error}; svd error={e}")
        return np.zeros(G.shape[0], dtype=float), "failed_zero_spectrum"

def response_spectrum_row(arch,seed,lr,mu,H,method,S,reference=None,optimizer="heavy_ball",beta2=np.nan,
                          memory_component="momentum",state_parameterization="additive",nominal_memory="reset",step_offset=0):
    r=len(S); G=torch.empty((r,r),dtype=torch.float64)
    for i in range(r):
        for j in range(r): G[i,j]=float(dot_state(S[i],S[j]))
    sing,spectrum_method=response_spectrum_singular_values_from_gram(G)
    energy=sing**2; total=float(energy.sum())
    probs=energy/total if total>0 else np.zeros_like(energy)
    entropy=float(-(probs[probs>0]*np.log(probs[probs>0])).sum()) if total>0 else np.nan
    cum=np.cumsum(probs) if total>0 else np.zeros_like(probs)
    row=dict(optimizer=optimizer,memory_component=memory_component,state_parameterization=state_parameterization,
             nominal_memory=nominal_memory,step_offset=int(step_offset),architecture=arch,seed=seed,horizon=H,
             learning_rate=lr,momentum=mu,beta2=beta2,method=method,response_rank=r,
             frobenius_norm=math.sqrt(total),spectral_norm=float(sing[0]) if len(sing) else np.nan,
             stable_rank=float(total/(sing[0]**2)) if len(sing) and sing[0]>0 else np.nan,
             effective_rank=math.exp(entropy) if np.isfinite(entropy) else np.nan,
             energy_top1=float(probs[0]) if len(probs) else np.nan,
             energy_top2=float(probs[:2].sum()) if len(probs) else np.nan,
             rank90=int(np.searchsorted(cum,0.9)+1) if total>0 else np.nan,
             spectrum_method=spectrum_method)
    for k,s in enumerate(sing, start=1): row[f"singular_value_{k}"]=float(s)
    if reference is not None:
        diff=[add_state(a,b,alpha=-1.0) for a,b in zip(S,reference)]
        row["relative_gap_to_time_varying"]=safe_ratio(columns_frobenius_norm(diff),columns_frobenius_norm(reference))
    return row


def save_application_response_tensor(arch,seed,lr,mu,H,S,tangent_seconds,optimizer="heavy_ball",beta2=np.nan,memory_component="momentum"):
    if optimizer in CFG.application_optimizers and memory_component=="momentum" and H==max(CFG.horizons):
        path=response_tensor_path(arch,seed,lr,mu,H,optimizer,beta2,memory_component)
        if FORCE_RESPONSE_GRID or not path.exists():
            torch_save_atomic({"S":cpu_state(S),"tangent_seconds":float(tangent_seconds),"optimizer":optimizer,
                               "beta2":beta2,"memory_component":memory_component},path)
            print(f"saved reusable RTTP tensor {path.name}")

def ensure_setting_columns(df, optimizer="heavy_ball", beta2=np.nan, memory_component="momentum",
                           state_parameterization="additive", nominal_memory="reset", step_offset=0):
    if "optimizer" not in df.columns: df["optimizer"]=optimizer
    if "beta2" not in df.columns: df["beta2"]=beta2
    if "memory_component" not in df.columns: df["memory_component"]=memory_component
    if "state_parameterization" not in df.columns: df["state_parameterization"]=state_parameterization
    if "nominal_memory" not in df.columns: df["nominal_memory"]=nominal_memory
    if "step_offset" not in df.columns: df["step_offset"]=int(step_offset)
    if "response_rank" not in df.columns:
        df["response_rank"]=2*CFG.response_rank if memory_component=="joint" else CFG.response_rank
    return df


def setting_identity(setting):
    return (setting["optimizer"],setting["memory_component"],float(setting["learning_rate"]),float(setting["momentum"]),
            float(setting["beta2"]) if np.isfinite(setting["beta2"]) else np.nan)


def load_all_setting_outputs(arch,seed,lr,mu,optimizer="heavy_ball",beta2=np.nan,memory_component="momentum",
                             state_parameterization="additive",nominal_memory="reset",step_offset=0):
    response_chunks=[]; spectrum_chunks=[]
    for H in CFG.horizons:
        chunk=read_dataframe(response_chunk_stem(arch,seed,lr,mu,H,optimizer,beta2,memory_component))
        spectrum=read_dataframe(spectrum_chunk_stem(arch,seed,lr,mu,H,optimizer,beta2,memory_component))
        if chunk is None or spectrum is None:
            return None
        chunk=ensure_setting_columns(chunk,optimizer,beta2,memory_component,state_parameterization,nominal_memory,step_offset)
        spectrum=ensure_setting_columns(spectrum,optimizer,beta2,memory_component,state_parameterization,nominal_memory,step_offset)
        missing=REQUIRED_RESPONSE_COLUMNS-set(chunk.columns)
        if missing:
            print(f"cached response chunk {response_chunk_stem(arch,seed,lr,mu,H,optimizer,beta2,memory_component).name} lacks new metrics {sorted(missing)}; recomputing setting")
            return None
        response_chunks.append(chunk); spectrum_chunks.append(spectrum)
    return response_chunks,spectrum_chunks

def run_response_setting(arch,seed,boundary,inherited,boundary_meta,future,probe,basis,alphas,lr,mu,optimizer="heavy_ball",beta2=np.nan):
    memory_component=basis["memory_component"]; state_parameterization=basis["state_parameterization"]; nominal_name=basis["nominal_memory"]
    step_offset=int(boundary_meta.get("step_count",0)) if optimizer in ("adam","adamw") else 0
    nominal_memory,nominal_meta=response_nominal_memory(boundary,inherited,optimizer,memory_component)
    nominal_name=nominal_meta["nominal_memory"]; state_parameterization=nominal_meta["state_parameterization"]
    B_coord=basis["coord"]; B_tangent=basis["tangent"]
    label=f"response opt={optimizer} mem={memory_component} arch={arch} seed={seed} lr={lr} mom/b1={mu}"
    rows_by_H={int(H):[] for H in CFG.horizons}; spectrum_by_H={int(H):[] for H in CFG.horizons}
    maxH=max(CFG.horizons); batches=future[:maxH]
    if optimizer in ("adam","adamw") and memory_component in ("log_v","joint"):
        assert memory_is_nonnegative_v(nominal_memory,boundary,optimizer), "nominal Adam v must be nonnegative"
    with timed_block(f"{label} RTTP maxH={maxH}"):
        S_tv_by_H,t_tv_by_H=tangent_response_time_varying_snapshots(boundary,batches,B_tangent,lr,mu,CFG.horizons,optimizer,beta2,nominal_memory,step_offset)
    save_application_response_tensor(arch,seed,lr,mu,maxH,S_tv_by_H[maxH],t_tv_by_H[maxH],optimizer,beta2,memory_component)

    S_fr_by_H={}; t_fr_by_H={}
    frozen_timer=EtaTimer(len(CFG.horizons),f"{label} frozen baselines")
    for H in CFG.horizons:
        t0=time.perf_counter()
        S_fr_by_H[H],t_fr_by_H[H]=tangent_response_frozen(boundary,future[:H],B_tangent,lr,mu,optimizer,beta2,nominal_memory,step_offset)
        spectrum_by_H[H].append(response_spectrum_row(arch,seed,lr,mu,H,"time_varying",S_tv_by_H[H],optimizer=optimizer,beta2=beta2,
                                                      memory_component=memory_component,state_parameterization=state_parameterization,
                                                      nominal_memory=nominal_name,step_offset=step_offset))
        spectrum_by_H[H].append(response_spectrum_row(arch,seed,lr,mu,H,"frozen_midpoint",S_fr_by_H[H],reference=S_tv_by_H[H],optimizer=optimizer,beta2=beta2,
                                                      memory_component=memory_component,state_parameterization=state_parameterization,
                                                      nominal_memory=nominal_name,step_offset=step_offset))
        frozen_timer.update(note=f"H={H}, step_time={format_duration(time.perf_counter()-t0)}")

    with timed_block(f"{label} nominal snapshots"):
        theta0_by_H,logits0_by_H=nonlinear_snapshots(boundary,batches,nominal_memory,lr,mu,CFG.horizons,probe,optimizer,beta2,step_offset)
    radius_base=component_radius_base(boundary,inherited,optimizer,memory_component)
    fd_timer=EtaTimer(len(alphas)*len(CFG.finite_difference_scales),f"{label} finite differences")
    for ai,alpha in enumerate(alphas):
        z_coord=combine_basis(B_coord,alpha); z_tangent=combine_basis(B_tangent,alpha); z_norm=state_norm_float(z_coord)
        pred_tv_by_H={H:combine_basis(S_tv_by_H[H],alpha) for H in CFG.horizons}
        pred_fr_by_H={H:combine_basis(S_fr_by_H[H],alpha) for H in CFG.horizons}
        pred_tv_norm_by_H={H:state_norm_float(pred_tv_by_H[H]) for H in CFG.horizons}
        pred_fr_norm_by_H={H:state_norm_float(pred_fr_by_H[H]) for H in CFG.horizons}
        frozen_gap_by_H={H:state_rel_error(pred_fr_by_H[H],pred_tv_by_H[H]) for H in CFG.horizons}
        for rs in CFG.finite_difference_scales:
            t_fd=time.perf_counter()
            eps=rs*radius_base
            mem_plus=memory_from_coordinate(boundary,nominal_memory,z_coord,eps,optimizer,memory_component)
            mem_minus=memory_from_coordinate(boundary,nominal_memory,z_coord,-eps,optimizer,memory_component)
            if optimizer in ("adam","adamw") and memory_component in ("log_v","joint"):
                assert memory_is_nonnegative_v(mem_plus,boundary,optimizer) and memory_is_nonnegative_v(mem_minus,boundary,optimizer)
            thp_by_H,logitsp_by_H=nonlinear_snapshots(boundary,batches,mem_plus,lr,mu,CFG.horizons,probe,optimizer,beta2,step_offset)
            thm_by_H,_=nonlinear_snapshots(boundary,batches,mem_minus,lr,mu,CFG.horizons,None,optimizer,beta2,step_offset)
            for H in CFG.horizons:
                actual=scale_state(add_state(thp_by_H[H],theta0_by_H[H],alpha=-1.0),1/eps)
                central=scale_state(add_state(thp_by_H[H],thm_by_H[H],alpha=-1.0),1/(2*eps))
                second=add_state(add_state(thp_by_H[H],theta0_by_H[H],alpha=-2.0),thm_by_H[H])
                actual_norm=state_norm_float(actual); central_norm=state_norm_float(central)
                cosine=state_cos(pred_tv_by_H[H],actual); central_cosine=state_cos(pred_tv_by_H[H],central)
                logit_disp=(logitsp_by_H[H]-logits0_by_H[H])/eps
                rows_by_H[H].append(dict(optimizer=optimizer,memory_component=memory_component,state_parameterization=state_parameterization,
                    nominal_memory=nominal_name,step_offset=step_offset,architecture=arch,seed=seed,horizon=H,learning_rate=lr,momentum=mu,beta2=beta2,
                    alpha_id=ai,radius_scale=rs,radius_base=radius_base,epsilon=eps,z_norm=z_norm,tangent_input_norm=state_norm_float(z_tangent),
                    true_norm=actual_norm,prediction_norm=pred_tv_norm_by_H[H],frozen_prediction_norm=pred_fr_norm_by_H[H],
                    central_norm=central_norm,response_gain=safe_ratio(pred_tv_norm_by_H[H],z_norm),
                    frozen_response_gain=safe_ratio(pred_fr_norm_by_H[H],z_norm),actual_gain=safe_ratio(actual_norm,z_norm),
                    endpoint_error=state_rel_error(pred_tv_by_H[H],actual),central_error=state_rel_error(pred_tv_by_H[H],central),
                    frozen_error=state_rel_error(pred_fr_by_H[H],actual),frozen_modeling_gap=frozen_gap_by_H[H],
                    cosine=cosine,angular_error=np.nan if not np.isfinite(cosine) else 1-cosine,
                    magnitude_error=abs(safe_ratio(pred_tv_norm_by_H[H],actual_norm)-1) if actual_norm>1e-12 else np.nan,
                    central_cosine=central_cosine,central_angular_error=np.nan if not np.isfinite(central_cosine) else 1-central_cosine,
                    central_magnitude_error=abs(safe_ratio(pred_tv_norm_by_H[H],central_norm)-1) if central_norm>1e-12 else np.nan,
                    second_difference_norm=state_norm_float(second),logit_displacement_norm=float(logit_disp.norm()),
                    tangent_seconds=t_tv_by_H[H],frozen_seconds=t_fr_by_H[H],degenerate=actual_norm<1e-10))
            del thp_by_H,logitsp_by_H,thm_by_H,mem_plus,mem_minus; gc.collect()
            fd_timer.update(note=f"alpha={ai}, radius={rs:g}, block_time={format_duration(time.perf_counter()-t_fd)}")
    return ({H:pd.DataFrame(rows) for H,rows in rows_by_H.items()},
            {H:pd.DataFrame(rows) for H,rows in spectrum_by_H.items()})

def response_condition_key(optimizer, memory_component, lr, mu, beta2):
    b2="nan" if not np.isfinite(float(beta2)) else float_tag(float(beta2))
    return (str(optimizer),str(memory_component),float_tag(float(lr)),float_tag(float(mu)),b2)

def expected_response_conditions():
    rows=[]
    for s in optimizer_settings():
        rows.append(response_condition_key(s["optimizer"],s["memory_component"],s["learning_rate"],s["momentum"],s["beta2"]))
    return set(rows)

def observed_response_conditions(df):
    needed={"optimizer","memory_component","learning_rate","momentum","beta2"}
    if df is None or not needed.issubset(df.columns): return set()
    rows=[]
    for _,r in df[["optimizer","memory_component","learning_rate","momentum","beta2"]].drop_duplicates().iterrows():
        rows.append(response_condition_key(r.optimizer,r.memory_component,r.learning_rate,r.momentum,r.beta2))
    return set(rows)

def run_response_grid(force: bool = False):
    chunks=[]; spectra=[]; maxH=max(CFG.horizons); settings=optimizer_settings()
    total_settings=len(CFG.architectures)*len(CFG.seeds)*len(settings)
    setting_timer=EtaTimer(total_settings,"response grid settings")
    for arch in CFG.architectures:
      for seed in CFG.seeds:
        with timed_block(f"data arch={arch} seed={seed}"):
            data=make_data(seed); future=materialize_batches(data["B_train"],maxH); probe=future[:max(1,min(4,len(future)))]
        for setting in settings:
          opt=setting["optimizer"]; lr=setting["learning_rate"]; mu=setting["momentum"]; beta2=setting["beta2"]
          mem=setting["memory_component"]; state_param=setting["state_parameterization"]; nominal_name=setting["nominal_memory"]
          with timed_block(f"boundary/load opt={opt} mem={mem} arch={arch} seed={seed}"):
              boundary,inherited,boundary_meta=train_or_load_boundary(arch,seed,data,opt)
              basis=make_component_basis(boundary,inherited,opt,mem,CFG.response_rank,seed)
              probe_offset=int(hashlib.md5(f"{opt}:{mem}".encode()).hexdigest()[:8],16)%10000
              rng=np.random.default_rng(seed+712+probe_offset)
              alphas=[]
              while len(alphas)<CFG.n_probe_directions:
                  a=rng.normal(size=basis["rank"])
                  n=np.linalg.norm(a)
                  if n>0: alphas.append(a/n)
              step_offset=int(boundary_meta.get("step_count",0)) if opt in ("adam","adamw") else 0
          t_setting=time.perf_counter()
          loaded=None if force else load_all_setting_outputs(arch,seed,lr,mu,opt,beta2,mem,state_param,nominal_name,step_offset)
          if loaded is not None:
              response_chunks,spectrum_chunks=loaded
              print(f"loaded response setting opt={opt} mem={mem} arch={arch} seed={seed} lr={lr} mom/b1={mu}: {sum(len(x) for x in response_chunks)} rows")
              chunks.extend(response_chunks); spectra.extend(spectrum_chunks)
              setting_timer.update(note=f"loaded opt={opt} mem={mem} arch={arch} seed={seed} lr={lr} mom/b1={mu}")
              continue
          setting_chunks,setting_spectra=run_response_setting(arch,seed,boundary,inherited,boundary_meta,future,probe,basis,alphas,lr,mu,opt,beta2)
          for H,chunk in setting_chunks.items():
              write_dataframe(chunk,response_chunk_stem(arch,seed,lr,mu,H,opt,beta2,mem))
              write_dataframe(setting_spectra[H],spectrum_chunk_stem(arch,seed,lr,mu,H,opt,beta2,mem))
              print(f"saved response chunk {response_chunk_stem(arch,seed,lr,mu,H,opt,beta2,mem).name}: {len(chunk)} rows")
              chunks.append(chunk); spectra.append(setting_spectra[H])
          setting_timer.update(note=f"computed opt={opt} mem={mem} arch={arch} seed={seed} lr={lr} mom/b1={mu}, setting_time={format_duration(time.perf_counter()-t_setting)}")
          gc.collect()
    df=pd.concat(chunks,ignore_index=True) if chunks else pd.DataFrame()
    spectrum=pd.concat(spectra,ignore_index=True) if spectra else pd.DataFrame()
    write_dataframe(df,TAB/"response_regime_raw"); write_dataframe(spectrum,TAB/"table_response_spectrum")
    return df,spectrum

response_raw = None if FORCE_RESPONSE_GRID else read_dataframe(TAB/"response_regime_raw")
response_spectrum = None if FORCE_RESPONSE_GRID else read_dataframe(TAB/"table_response_spectrum")
if response_raw is not None and "optimizer" not in response_raw.columns:
    print("full response_regime_raw is heavy-ball-only legacy output; rebuilding combined table from chunks without recomputing heavy-ball")
    response_raw=None; response_spectrum=None
elif response_raw is not None and "memory_component" not in response_raw.columns:
    print("full response_regime_raw contains legacy adaptive outputs without memory_component; rebuilding from component-aware chunks")
    response_raw=None; response_spectrum=None
elif response_raw is not None and (REQUIRED_RESPONSE_COLUMNS-set(response_raw.columns) or not expected_response_conditions().issubset(observed_response_conditions(response_raw))):
    print("full response_regime_raw lacks required columns or optimizer/memory-component conditions; rebuilding/computing missing settings")
    response_raw=None; response_spectrum=None
if response_raw is None or response_spectrum is None:
    response_raw,response_spectrum = run_response_grid(force=FORCE_RESPONSE_GRID)
else:
    print(f"loaded full response_regime_raw: {len(response_raw)} rows")
    print(f"loaded full table_response_spectrum: {len(response_spectrum)} rows")
response_raw.head()


In [ ]:
def ci95(x):
    x=pd.Series(x).dropna().to_numpy(float)
    if len(x)<2: return (float(np.mean(x)) if len(x) else np.nan,np.nan,np.nan)
    m=float(x.mean()); h=float(stats.t.ppf(.975,len(x)-1)*x.std(ddof=1)/math.sqrt(len(x)))
    return m,m-h,m+h

def loglog_slope(g,col):
    sub=g[["epsilon",col]].replace([np.inf,-np.inf],np.nan).dropna()
    sub=sub[(sub.epsilon>0)&(sub[col]>0)]
    if len(sub)<2: return np.nan
    return float(np.polyfit(np.log(sub.epsilon.to_numpy(float)),np.log(sub[col].to_numpy(float)),1)[0])

response_raw=ensure_setting_columns(response_raw,"heavy_ball",np.nan,"momentum","additive","reset",0)
response_spectrum=ensure_setting_columns(response_spectrum,"heavy_ball",np.nan,"momentum","additive","reset",0)
condition_cols=["optimizer","memory_component","architecture","seed","horizon","learning_rate","momentum","beta2","state_parameterization","nominal_memory","step_offset"]
summary_cols=["optimizer","memory_component","architecture","horizon","learning_rate","momentum","beta2","state_parameterization","nominal_memory"]
linearity_cols=condition_cols+["alpha_id"]
radius_condition_cols=["optimizer","memory_component"]

radius_score=(response_raw[~response_raw.degenerate].groupby(radius_condition_cols+["radius_scale"],as_index=False)
              .central_error.median().rename(columns={"central_error":"median_central_error"}))
primary_radius=(radius_score.sort_values("median_central_error")
                .groupby(radius_condition_cols,as_index=False).first()
                .rename(columns={"radius_scale":"primary_radius_scale"}))
PRIMARY_RADIUS=float(primary_radius[(primary_radius.optimizer=="heavy_ball")&(primary_radius.memory_component=="momentum")].primary_radius_scale.iloc[0]) if len(primary_radius[(primary_radius.optimizer=="heavy_ball")&(primary_radius.memory_component=="momentum")]) else float(primary_radius.primary_radius_scale.iloc[0])
print("Primary heavy-ball calibrated radius scale:",PRIMARY_RADIUS)
write_dataframe(radius_score,TAB/"radius_score")
write_dataframe(primary_radius,TAB/"table_primary_radius_by_memory")

response_with_radius=response_raw.merge(primary_radius[radius_condition_cols+["primary_radius_scale"]],on=radius_condition_cols,how="left")
local=response_with_radius[(response_with_radius.radius_scale==response_with_radius.primary_radius_scale)&(~response_with_radius.degenerate)].copy()
metric_cols=["endpoint_error","frozen_error","cosine","tangent_seconds","response_gain","actual_gain",
             "frozen_modeling_gap","magnitude_error","angular_error","prediction_norm","true_norm",
             "radius_base","tangent_input_norm"]
seed_level=(local.groupby(condition_cols,as_index=False,dropna=False)
            .agg(**{col:(col,"mean") for col in metric_cols if col in local.columns},
                 primary_radius_scale=("primary_radius_scale","first")))
write_dataframe(seed_level,TAB/"table_seed_level_response")
summary=[]
for keys,g in seed_level.groupby(summary_cols,dropna=False):
    row=dict(zip(summary_cols,keys))
    row["step_offset_min"]=int(g.step_offset.min()) if "step_offset" in g else 0
    row["step_offset_max"]=int(g.step_offset.max()) if "step_offset" in g else 0
    row["primary_radius_scale"]=float(g.primary_radius_scale.iloc[0]) if "primary_radius_scale" in g else np.nan
    for col in [c for c in metric_cols if c in seed_level.columns]:
        m,lo,hi=ci95(g[col]); row.update({col:m,f"{col}_lo":lo,f"{col}_hi":hi})
    row["improvement_over_frozen"]=row["frozen_error"]/max(row["endpoint_error"],1e-12)
    summary.append(row)
summary=pd.DataFrame(summary)
write_dataframe(summary,TAB/"table_main_response")

linearity_rows=[]
for keys,g in response_raw[~response_raw.degenerate].groupby(linearity_cols,dropna=False):
    row=dict(zip(linearity_cols,keys))
    row["central_error_slope"]=loglog_slope(g,"central_error")
    row["endpoint_error_slope"]=loglog_slope(g,"endpoint_error")
    row["second_difference_slope"]=loglog_slope(g,"second_difference_norm")
    row["magnitude_error_slope"]=loglog_slope(g,"magnitude_error")
    linearity_rows.append(row)
linearity=pd.DataFrame(linearity_rows)
write_dataframe(linearity,TAB/"table_linearity_slopes_raw")
linearity_summary=(linearity.groupby(summary_cols,as_index=False,dropna=False)
                   .agg(central_error_slope=("central_error_slope","mean"),
                        endpoint_error_slope=("endpoint_error_slope","mean"),
                        second_difference_slope=("second_difference_slope","mean"),
                        magnitude_error_slope=("magnitude_error_slope","mean")))
write_dataframe(linearity_summary,TAB/"table_linearity_slopes")

spectrum_summary=(response_spectrum.groupby(summary_cols+["method"],as_index=False,dropna=False)
                  .agg(frobenius_norm=("frobenius_norm","mean"),spectral_norm=("spectral_norm","mean"),
                       stable_rank=("stable_rank","mean"),effective_rank=("effective_rank","mean"),
                       energy_top1=("energy_top1","mean"),energy_top2=("energy_top2","mean"),
                       rank90=("rank90","mean"),relative_gap_to_time_varying=("relative_gap_to_time_varying","mean"),
                       response_rank=("response_rank","mean")))
write_dataframe(spectrum_summary,TAB/"table_response_spectrum_summary")

adaptive_summary=summary[summary.optimizer.isin(["adam","adamw"])].copy()
write_dataframe(adaptive_summary,TAB/"table_adaptive_response_summary")
adaptive_spectrum=spectrum_summary[(spectrum_summary.optimizer.isin(["adam","adamw"]))&(spectrum_summary.method=="time_varying")].copy()
write_dataframe(adaptive_spectrum,TAB/"table_adaptive_response_spectrum")
summary.head()


In [ ]:
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG/f"{name}.pdf",bbox_inches="tight"); plt.savefig(FIG/f"{name}.png",dpi=220,bbox_inches="tight"); plt.show(); plt.close()

def selected_summary_rows(opt="heavy_ball", memory_component=None):
    if opt=="heavy_ball":
        mem="momentum" if memory_component is None else memory_component
        return summary[(summary.optimizer==opt)&(summary.memory_component==mem)&(summary.learning_rate==CFG.response_lrs[-1])&(summary.momentum==CFG.response_momenta[-1])]
    mem="m" if memory_component is None else memory_component
    return summary[(summary.optimizer==opt)&(summary.memory_component==mem)&(summary.learning_rate==CFG.adam_response_lrs[-1])&(summary.momentum==CFG.adam_beta1s[-1])]

for opt,mem in sorted(summary[["optimizer","memory_component"]].drop_duplicates().itertuples(index=False,name=None)):
  for arch in CFG.architectures:
    sub=selected_summary_rows(opt,mem)
    sub=sub[sub.architecture==arch]
    if not len(sub): continue
    plt.figure(figsize=(5.2,3.5))
    plt.plot(sub.horizon,sub.endpoint_error,marker="o",label="Time-varying tangent")
    plt.fill_between(sub.horizon,sub.endpoint_error_lo,sub.endpoint_error_hi,alpha=.2)
    plt.plot(sub.horizon,sub.frozen_error,marker="s",label="Frozen midpoint")
    plt.fill_between(sub.horizon,sub.frozen_error_lo,sub.frozen_error_hi,alpha=.2)
    plt.yscale("log"); plt.xlabel("Horizon H"); plt.ylabel("Endpoint relative error"); plt.title(f"{arch} / {opt} / {mem}")
    plt.legend(frameon=False); savefig(f"fig_endpoint_error_{opt}_{mem}_{arch}")

plt.figure(figsize=(5.2,3.5))
sc=(response_raw[~response_raw.degenerate].groupby(["optimizer","memory_component","architecture","radius_scale"],as_index=False)
    .central_error.median())
write_dataframe(sc,TAB/"table_finite_difference_regime")
for (opt,mem,arch),g in sc.groupby(["optimizer","memory_component","architecture"]):
    plt.plot(g.radius_scale,g.central_error,marker="o",label=f"{arch}/{opt}/{mem}")
plt.xscale("log"); plt.yscale("log"); plt.xlabel("Radius scale"); plt.ylabel("Median tangent-central difference error")
plt.legend(frameon=False); savefig("fig_finite_difference_regime")

plt.figure(figsize=(5.2,3.5))
for opt,mem in sorted(summary[["optimizer","memory_component"]].drop_duplicates().itertuples(index=False,name=None)):
  for arch,g in selected_summary_rows(opt,mem).groupby("architecture"):
    plt.plot(g.horizon,g.response_gain,marker="o",label=f"{arch}/{opt}/{mem}")
plt.yscale("log"); plt.xlabel("Horizon H"); plt.ylabel("Mean response gain")
plt.legend(frameon=False); savefig("fig_response_gain")

plt.figure(figsize=(5.2,3.5))
for opt,mem in sorted(summary[["optimizer","memory_component"]].drop_duplicates().itertuples(index=False,name=None)):
  for arch,g in selected_summary_rows(opt,mem).groupby("architecture"):
    plt.plot(g.horizon,g.frozen_modeling_gap,marker="o",label=f"{arch}/{opt}/{mem}")
plt.yscale("log"); plt.xlabel("Horizon H"); plt.ylabel("Frozen-vs-RTTP operator gap")
plt.legend(frameon=False); savefig("fig_frozen_modeling_gap")

plt.figure(figsize=(5.2,3.5))
merge_keys=["optimizer","memory_component","architecture","horizon","learning_rate","momentum","beta2","state_parameterization","nominal_memory"]
for opt,mem in sorted(summary[["optimizer","memory_component"]].drop_duplicates().itertuples(index=False,name=None)):
    ss=selected_summary_rows(opt,mem)[merge_keys].merge(
        spectrum_summary[spectrum_summary.method=="time_varying"],
        on=merge_keys,
        how="inner",
    )
    for arch,g in ss.groupby("architecture"):
        plt.plot(g.horizon,g.effective_rank,marker="o",label=f"{arch}/{opt}/{mem}")
plt.xlabel("Horizon H"); plt.ylabel("Effective response rank")
plt.legend(frameon=False); savefig("fig_effective_response_rank")


In [ ]:
rank_rows=[]
for arch in CFG.architectures:
    d=n_parameters(build_model(arch))
    for r in CFG.rank_sweep:
        rank_rows.append(dict(architecture=arch,d=d,rank=r,
            dense_response_gib=d*d*4/2**30,
            reduced_basis_response_gib=2*d*r*4/2**30,
            tangent_working_state_gib=2*d*r*4/2**30,
            compression_ratio=(d*d)/(2*d*r)))
rank_table=pd.DataFrame(rank_rows)
write_dataframe(rank_table,TAB/"table_rank_storage")
atomic_write_text(TAB/"table_rank_storage.tex", rank_table.to_latex(index=False,float_format="%.3g"))
rank_table


In [ ]:
def projected_loss_quadratic(endpoint_model, probe_batch, S):
    x,y=probe_batch; x,y=x.to(DEVICE),y.to(DEVICE)
    loss,g=loss_and_grads(endpoint_model,x,y,create_graph=True)
    q=torch.tensor([dot_state(g,s) for s in S],device=DEVICE)
    r=len(S); Q=torch.empty(r,r,device=DEVICE)
    for j,s in enumerate(S):
        hs=torch.autograd.grad(dot_state(g,s),params_list(endpoint_model),retain_graph=True)
        for i,si in enumerate(S): Q[i,j]=dot_state(si,hs)
    Q=(Q+Q.T)/2
    ev,U=torch.linalg.eigh(Q); Qpsd=(U*torch.clamp(ev,min=0))@U.T
    return float(loss.detach()),q.detach(),Qpsd.detach()

def solve_ball_quadratic(q,Q,radius):
    
    q=q.double(); Q=Q.double(); I=torch.eye(len(q),device=q.device,dtype=q.dtype)
    def sol(lam): return torch.linalg.solve(Q+lam*I,-q)
    a=sol(1e-10)
    if float(a.norm())<=radius: return a.float()
    lo,hi=0.0,1.0
    while float(sol(hi).norm())>radius: hi*=2
    for _ in range(60):
        mid=(lo+hi)/2
        if float(sol(mid).norm())>radius: lo=mid
        else: hi=mid
    return sol(hi).float()

def run_policy(boundary,batches,z,lr,mu,A_test,B_test,optimizer="heavy_ball",beta2=np.nan):
    _,model=nonlinear_endpoint(boundary,batches,z,lr,mu,optimizer,beta2)
    a=evaluate(model,A_test); b=evaluate(model,B_test)
    return {"A_loss":a["loss"],"A_acc":a["acc"],"B_loss":b["loss"],"B_acc":b["acc"]}

def load_saved_application_response(arch,seed,lr,mu,H,optimizer="heavy_ball",beta2=np.nan):
    path=response_tensor_path(arch,seed,lr,mu,H,optimizer,beta2)
    if not path.exists():
        return None
    try:
        payload=torch.load(path,map_location="cpu")
        print(f"loaded reusable RTTP tensor {path.name}")
        return device_state(payload["S"])
    except Exception as e:
        print(f"could not load reusable RTTP tensor {path.name}; recomputing: {e}")
        return None

def run_application(force: bool = False):
    chunks=[]
    if not CFG.run_application: return pd.DataFrame()
    H=max(CFG.horizons)
    app_settings=[s for s in optimizer_settings() if s["optimizer"] in CFG.application_optimizers and s["memory_component"]=="momentum"]
    total=len(CFG.architectures)*len(CFG.seeds)*len(app_settings)
    app_timer=EtaTimer(total,"application policy chunks")
    for arch in CFG.architectures:
      for seed in CFG.seeds:
        data=None
        for setting in app_settings:
          opt=setting["optimizer"]; lr=setting["learning_rate"]; mu=setting["momentum"]; beta2=setting["beta2"]
          t_chunk=time.perf_counter()
          stem=policy_chunk_stem(arch,seed,opt,lr,mu,beta2)
          cached=None if force else read_dataframe(stem)
          if cached is not None:
              cached=ensure_setting_columns(cached,opt,beta2)
              print(f"loaded policy chunk {stem.name}: {len(cached)} rows")
              chunks.append(cached)
              app_timer.update(note=f"loaded opt={opt} arch={arch} seed={seed}")
              continue

          rows=[]
          with timed_block(f"application data/boundary opt={opt} arch={arch} seed={seed}"):
              if data is None: data=make_data(seed)
              boundary,inherited,_=train_or_load_boundary(arch,seed,data,opt)
              batches=materialize_batches(data["B_train"],H)
              B=make_basis(memory_template_like(boundary,opt),inherited,CFG.response_rank,seed)
          S=None if force else load_saved_application_response(arch,seed,lr,mu,H,opt,beta2)
          if S is None:
              with timed_block(f"application RTTP opt={opt} arch={arch} seed={seed} H={H}"):
                  S,_=tangent_response_time_varying(boundary,batches,B,lr,mu,opt,beta2)
              torch_save_atomic({"S":cpu_state(S),"optimizer":opt,"beta2":beta2},response_tensor_path(arch,seed,lr,mu,H,opt,beta2))
              print(f"saved reusable RTTP tensor {response_tensor_path(arch,seed,lr,mu,H,opt,beta2).name}")
          with timed_block(f"application certificate opt={opt} arch={arch} seed={seed}"):
              _,endpoint=nonlinear_endpoint(boundary,batches,memory_zeros_like(boundary,opt),lr,mu,opt,beta2)
              Aprobe=next(iter(data["A_test"])); Bprobe=next(iter(data["B_test"]))
              _,qA,QA=projected_loss_quadratic(endpoint,Aprobe,S); _,qB,QB=projected_loss_quadratic(endpoint,Bprobe,S)
              base_radius=max(float(norm_state(inherited)),1e-8)
              candidates=[]
              for trade in CFG.certificate_tradeoffs:
                for rs in CFG.certificate_radius_scales:
                  alpha=solve_ball_quadratic(qA+trade*qB,QA+trade*QB,rs*base_radius)
                  predA=float(qA@alpha+.5*alpha@QA@alpha); predB=float(qB@alpha+.5*alpha@QB@alpha)
                  candidates.append((predA+trade*predB,trade,rs,alpha,predA,predB))
              _,trade,rs,alpha,predA,predB=min(candidates,key=lambda t:t[0])
          policies={"reset":memory_zeros_like(boundary,opt),"inherit":inherited,
                    "tangent_certificate":combine_basis(B,alpha.cpu().numpy())}
          rng=np.random.default_rng(seed+991); ar=rng.normal(size=len(B)); ar=ar/np.linalg.norm(ar)*float(np.linalg.norm(alpha.cpu()))
          policies["random_matched_norm"]=combine_basis(B,ar)
          policy_timer=EtaTimer(len(policies),f"application policies opt={opt} arch={arch} seed={seed}")
          for name,z in policies.items():
              t_policy=time.perf_counter()
              met=run_policy(boundary,batches,z,lr,mu,data["A_test"],data["B_test"],opt,beta2)
              rows.append(dict(optimizer=opt,architecture=arch,seed=seed,policy=name,horizon=H,learning_rate=lr,momentum=mu,beta2=beta2,
                               certificate_tradeoff=trade,certificate_radius_scale=rs,predicted_A_change=predA,
                               predicted_B_change=predB,**met))
              policy_timer.update(note=f"policy={name}, policy_time={format_duration(time.perf_counter()-t_policy)}")
          chunk=pd.DataFrame(rows)
          write_dataframe(chunk,stem)
          print(f"saved policy chunk {stem.name}: {len(chunk)} rows")
          chunks.append(chunk)
          del S,endpoint; gc.collect()
          app_timer.update(note=f"computed opt={opt} arch={arch} seed={seed}, chunk_time={format_duration(time.perf_counter()-t_chunk)}")
    out=pd.concat(chunks,ignore_index=True) if chunks else pd.DataFrame()
    out=out.drop(columns=["A_loss_reset","B_loss_reset","delta_forgetting","delta_progress"],errors="ignore")
    reset=out[out.policy=="reset"][["optimizer","architecture","seed","A_loss","B_loss"]].rename(columns={"A_loss":"A_loss_reset","B_loss":"B_loss_reset"})
    out=out.merge(reset,on=["optimizer","architecture","seed"]); out["delta_forgetting"]=out.A_loss-out.A_loss_reset; out["delta_progress"]=out.B_loss_reset-out.B_loss
    write_dataframe(out,TAB/"policy_results")
    return out

policy_results = None if FORCE_APPLICATION else read_dataframe(TAB/"policy_results")
if policy_results is None or (len(policy_results) and "optimizer" not in policy_results.columns):
    if policy_results is not None and "optimizer" not in policy_results.columns:
        print("full policy_results lacks optimizer column; rebuilding from chunks/application settings")
    policy_results=run_application(force=FORCE_APPLICATION)
else:
    print(f"loaded full policy_results: {len(policy_results)} rows")
policy_results.head() if len(policy_results) else policy_results


In [ ]:
if len(policy_results):
    policy_results=ensure_setting_columns(policy_results,"heavy_ball",np.nan)
    policy_summary=(policy_results.groupby(["optimizer","architecture","policy"],as_index=False,dropna=False)
                    .agg(delta_forgetting=("delta_forgetting","mean"),delta_progress=("delta_progress","mean"),
                         A_acc=("A_acc","mean"),B_acc=("B_acc","mean")))
    write_dataframe(policy_summary,TAB/"table_policy_summary")
    atomic_write_text(TAB/"table_policy_summary.tex", policy_summary.to_latex(index=False,float_format="%.4f"))
    display(policy_summary)


In [ ]:
checks=[]
def check(name,passed,detail): checks.append(dict(check=name,passed=bool(passed),detail=str(detail)))
check("finite_core_metrics",np.isfinite(response_raw[["true_norm","second_difference_norm","tangent_seconds"]]).all().all(),"core finite")
check("scientific_object_columns",REQUIRED_RESPONSE_COLUMNS.issubset(response_raw.columns),sorted(REQUIRED_RESPONSE_COLUMNS-set(response_raw.columns)))
check("complete_architectures",set(response_raw.architecture)==set(CFG.architectures),sorted(response_raw.architecture.unique()))
check("complete_optimizers",set(CFG.optimizers).issubset(set(response_raw.optimizer)),sorted(response_raw.optimizer.unique()))
expected_components={"heavy_ball":{"momentum"}}
for opt in CFG.optimizers:
    if opt in ("adam","adamw"):
        expected_components[opt]=set(CFG.adaptive_memory_components)
observed_components={opt:set(g.memory_component.astype(str)) for opt,g in response_raw.groupby("optimizer")}
check("complete_memory_components",all(expected_components.get(opt,set()).issubset(observed_components.get(opt,set())) for opt in expected_components),observed_components)
check("nondegenerate_primary",(~local.degenerate if "degenerate" in local else pd.Series([True])).all(),f"n={len(local)}")
heavy_local=local[(local.optimizer=="heavy_ball")&(local.memory_component=="momentum")]
check("heavy_ball_time_varying_beats_frozen",float((heavy_local.endpoint_error<heavy_local.frozen_error).mean())>.5,
      f"fraction={float((heavy_local.endpoint_error<heavy_local.frozen_error).mean()):.3f}")
check("cosine_in_range",local.cosine.dropna().between(-1.0001,1.0001).all(),
      f"range=[{local.cosine.min():.4f},{local.cosine.max():.4f}]")
small_large=[]
for (opt,mem),g in response_raw[response_raw.optimizer.isin(["adam","adamw"]) & response_raw.memory_component.isin(["m","log_v"]) & (~response_raw.degenerate)].groupby(["optimizer","memory_component"]):
    med=g.groupby("radius_scale").central_error.median().sort_index()
    if len(med): small_large.append((opt,mem,float(med.iloc[0]),float(med.iloc[-1]),float(med.iloc[0])<float(med.iloc[-1])))
check("adaptive_small_radius_central_error",all(x[-1] for x in small_large),small_large)
check("spectrum_available",len(response_spectrum)>0 and {"time_varying","frozen_midpoint"}.issubset(set(response_spectrum.method)),
      f"n={len(response_spectrum)} methods={sorted(response_spectrum.method.unique()) if len(response_spectrum) else []}")
check("task_A_accuracy_nonzero",True if not len(policy_results) else float(policy_results.A_acc.max())>0,
      "label remapping/evaluation sanity")
quality=pd.DataFrame(checks); write_dataframe(quality,TAB/"quality_report")
quality


In [ ]:
def pm(m,lo,hi):
    if not np.isfinite(lo): return f"{m:.3g}"
    return rf"{m:.3g} $\pm$ {(hi-lo)/2:.2g}"
main=selected_summary_rows("heavy_ball","momentum").copy()
main["RTTP error"]=main.apply(lambda r:pm(r.endpoint_error,r.endpoint_error_lo,r.endpoint_error_hi),axis=1)
main["Frozen error"]=main.apply(lambda r:pm(r.frozen_error,r.frozen_error_lo,r.frozen_error_hi),axis=1)
main["Gain"]=main.apply(lambda r:pm(r.response_gain,r.response_gain_lo,r.response_gain_hi),axis=1)
main["Frozen gap"]=main.apply(lambda r:pm(r.frozen_modeling_gap,r.frozen_modeling_gap_lo,r.frozen_modeling_gap_hi),axis=1)
latex=main[["optimizer","memory_component","architecture","horizon","RTTP error","Frozen error","Gain","Frozen gap","improvement_over_frozen"]].to_latex(index=False,escape=False,float_format="%.2f")
atomic_write_text(TAB/"table_main_response.tex", latex)
print(latex)

optimizer_comp=summary.copy()
optimizer_comp["RTTP error"]=optimizer_comp.apply(lambda r:pm(r.endpoint_error,r.endpoint_error_lo,r.endpoint_error_hi),axis=1)
optimizer_comp["Frozen error"]=optimizer_comp.apply(lambda r:pm(r.frozen_error,r.frozen_error_lo,r.frozen_error_hi),axis=1)
latex_opt=optimizer_comp[["optimizer","memory_component","state_parameterization","nominal_memory","architecture","horizon","learning_rate","momentum","beta2","primary_radius_scale","RTTP error","Frozen error","improvement_over_frozen"]].to_latex(index=False,escape=False,float_format="%.4g")
atomic_write_text(TAB/"table_optimizer_comparison.tex", latex_opt)
print(latex_opt[:4000])


In [ ]:
manifest={
    "created_utc":pd.Timestamp.utcnow().isoformat(),"device":str(DEVICE),"torch":torch.__version__,
    "python":sys.version,"platform":platform.platform(),"config":asdict(CFG),
    "root":str(ROOT),"use_google_drive":bool(RUN_IN_COLAB and USE_GOOGLE_DRIVE),
}
atomic_write_text(ROOT/"manifest.json", json.dumps(manifest,indent=2,default=str))
try:
    atomic_write_text(ROOT/"pip_freeze.txt", subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True))
except Exception as e:
    atomic_write_text(ROOT/"pip_freeze.txt", str(e))
for candidate in Path.cwd().glob("*.ipynb"):
    if "optimizer_response" in candidate.name:
        shutil.copy2(candidate,ROOT/candidate.name)
zip_path=ARCH/f"optimizer_response_cifar10_rttp_{pd.Timestamp.utcnow().strftime('%Y%m%d_%H%M%S')}.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as zf:
    for p in ROOT.rglob("*"):
        if p.is_file() and ARCH not in p.parents:
            zf.write(p,p.relative_to(ROOT.parent))
print("output root:", ROOT)
print("archive:", zip_path)
try:
    if RUN_IN_COLAB:
        from google.colab import files
        files.download(str(zip_path))
    else:
        from IPython.display import FileLink,display
        display(FileLink(str(zip_path)))
except Exception as e:
    print("download helper skipped; archive is already persisted at:", zip_path, "reason:", e)
